# Automated Drought Impact Extraction — Full Pipeline

This notebook takes **one file of newspaper articles** and runs the complete pipeline,
from raw text to classified drought-impact sentences with locations.

You do not need to write any code. You only edit **Block 1 (Settings)**, then run the
cells from top to bottom.

## Two ways to run it: TRAIN and APPLY

The notebook has a single switch in Block 1, `MODE`, with two values:

- **`MODE = "train"`** — Learn the models from an annotated corpus (currently the
  1990–2024 articles). This runs the training steps and saves three models to
  `models/`. **You do this once.**
- **`MODE = "apply"`** — Take those saved models and run them on **new, un-annotated**
  articles (for example 2025 onward). The training steps are skipped and the models are
  loaded from disk. You do this for each new batch, and it needs no annotations at all.

If you have already annotated older articles and now just want to classify newer ones,
run once in `"train"` mode on the old corpus, then switch to `"apply"` for the new files.
You do **not** need to re-process the historical articles.

## What each block does

| Block | Stage | Runs in | Output file |
|---|---|---|---|
| 0 | Environment check | both | — |
| 1 | Settings — the only place you edit | both | — |
| 2 | Setup + file check | both | — |
| 3 | Filtering: clean, de-duplicate, drop foreign/irrelevant | both | `05_filtered.csv` |
| 4 | Load annotations | train only | `annotations_combined.csv` |
| 5a / 5b | Article relevance: **train+save** / **apply** | 5a train, 5b both | `06_article_relevance.csv` |
| 6 | Sentence preparation (split; match spans in train mode) | both | `07_sentences.csv` |
| 7a / 7b | Sentence relevance: **train+save** / **apply** | 7a train, 7b both | `08_sentence_relevance.csv` |
| 8a / 8b | Impact classification: **train+save** / **apply** | 8a train, 8b both | `09_impact_predictions.csv` |
| 9 | Pipeline evaluation | train only | printed report |
| 10 | Location extraction (geoparsing) | both | `11_location_clusters.csv` |
| 11 | Summary | both | — |

The training-only blocks skip themselves automatically in apply mode. It is still possible to run
every cell top to bottom, the skipped ones just print a short note and do nothing.

Two optional appendices at the end show how to **prepare a new batch of articles for annotation**
and **measure agreement between two annotators**.

## How long does it take?

The two geoparsing steps (Block 3 and Block 10) and, in train mode, the sentence-model
search (Block 7a) are by far the slowest parts. For a large corpus expect several hours.
Apply mode is much faster, since no training happens, the time is mostly the two
geoparsing passes. Every block saves its result to a file, so you can stop after any
block, close the notebook, and continue later. You just need to re-run Block 1 and Block 2 first,
then carry on where you stopped.

## General tips

- **Run the blocks in order the first time.** After that, any block can be re-run on its own.
- **If a block fails**, read the message printed in red. Almost all errors are a missing
  file or a missing package, and the message says which one.
- **Nothing is ever overwritten in your input folders.** All results go to the output folder.

## Files you need before you start

This notebook needs to be in a folder that looks like this. The names in **bold** are required;
the rest are optional.

```
my_project/
├── drought_pipeline.ipynb          <- this notebook
├── requirements.txt
│
├── input/
│   └── DE_articles.tsv             <- **REQUIRED**: news articles from swissdox query (.tsv or .csv)
│
├── annotation/
│   ├── DE_annotated_impacts_locations.csv   <- **REQUIRED**: span annotations
│   ├── DE_annotated_impacts_batch2.csv      <- optional: more annotations, added later
│   ├── DE_annotated_irrelevant.csv          <- **REQUIRED**: articles judged irrelevant
│   ├── DE_annotated_relevant.csv            <- optional: extra relevant article IDs
│   └── DE_annotated_foreign.csv             <- optional: only for checking the Swiss filter
│
└── output/                          <- created automatically, all results land here
```

### 1. The article file (`input/`)

A `.tsv` (tab-separated) or `.csv` file with one row per article. These columns must exist:

| Column | Meaning |
|---|---|
| `content_id` | A unique ID for the article |
| `pubtime` | Publication date/time |
| `head` | Headline |
| `subhead` | Sub-headline (may be empty) |
| `content` | Article body text |
| `rubric` | Newspaper section, e.g. "Inland", "Monde" (optional) |

### 2. The annotation files (`annotation/`)

These come out of the annotation tool as CSV exports. **It is possible to have multiple
of these.** The notebook finds them by name pattern, so new annotated batches should be
saved into the `annotation/` folder with a matching name and re-run.

| Pattern | Contains | Used for |
|---|---|---|
| `{LANG}_annotated_impacts*.csv` | Span annotations: `FILE`, `ANCHOR`, `TAGS`, `QUOTE_TRANSCRIPTION`, and for locations `VOCAB_LABEL`, `LAT`, `LNG`, `COMMENTS` | Training the sentence and impact models; the articles listed here count as "relevant" |
| `{LANG}_annotated_irrelevant*.csv` | A `FILE` column of article IDs judged not about drought | Negative examples for the article-relevance model |
| `{LANG}_annotated_relevant*.csv` | A `FILE` column of article IDs judged about drought (no spans) | Extra positive examples |
| `{LANG}_annotated_foreign*.csv` | A `FILE` column plus `COMMENTS` | Optional check of the Swiss/foreign filter |

So for example `DE_annotated_impacts_locations.csv`, `DE_annotated_impacts_round2.csv` and
`DE_annotated_impacts_2026_january.csv` are all picked up automatically and merged.

> **Note on adding more annotated data.** Annotation itself happens outside this notebook,
> in the annotation tool (Recogito). To grow the training set: run **Appendix A** at the bottom to
> export a fresh sample of articles as TEI files, annotate them in Recogito, export the
> result as CSV, drop it into `annotation/`, and re-run Blocks 4 to 9. Duplicate rows
> across files are removed automatically, so re-exporting an old batch does no harm.

---
# Block 0 — Environment check

Run this first. It does not change anything; it only tells you whether everything the
pipeline needs is installed. If something is missing, the exact install command is printed.

In [2]:
# ============================================================================
# BLOCK 0 — ENVIRONMENT CHECK
# ============================================================================
# This cell checks for required Python packages and spaCy language models.
# It never installs anything by itself, so it is always safe to run.

import importlib.util
import subprocess
import sys

# Package name as you import it  ->  name you install it with
REQUIRED_PACKAGES = {
    "pandas":               "pandas",
    "numpy":                "numpy",
    "sklearn":              "scikit-learn",
    "scipy":                "scipy",
    "joblib":               "joblib",
    "tqdm":                 "tqdm",
    "datasketch":           "datasketch",          # near-duplicate detection
    "spacy":                "spacy",               # sentence splitting
    "geoparser":            "geoparser==0.2.3",    # place-name recognition
    "sentence_transformers": "sentence-transformers",  # multilingual embeddings
    "iterstrat":            "iterative-stratification",  # multi-label data splits
    "lxml":                 "lxml",                # TEI export (Appendix A only)
}

# Optional: only needed for the canton-level location check in Block 10
OPTIONAL_PACKAGES = {
    "geopandas": "geopandas",
    "shapely":   "shapely",
}

# spaCy models, one per language
SPACY_MODELS = {
    "DE": "de_core_news_lg",
    "FR": "fr_core_news_lg",
    "IT": "it_core_news_lg",
}

print("=" * 70)
print("PYTHON")
print("=" * 70)
print(sys.version.split()[0], "at", sys.executable)

print()
print("=" * 70)
print("REQUIRED PACKAGES")
print("=" * 70)

missing = []
for module_name, install_name in REQUIRED_PACKAGES.items():
    found = importlib.util.find_spec(module_name) is not None
    print(f"  {'OK     ' if found else 'MISSING'}  {module_name}")
    if not found:
        missing.append(install_name)

print()
print("=" * 70)
print("OPTIONAL PACKAGES (canton-level location check only)")
print("=" * 70)

for module_name, install_name in OPTIONAL_PACKAGES.items():
    found = importlib.util.find_spec(module_name) is not None
    print(f"  {'OK     ' if found else 'missing'}  {module_name}")

print()
print("=" * 70)
print("SPACY LANGUAGE MODELS")
print("=" * 70)

missing_models = []
if importlib.util.find_spec("spacy") is not None:
    import spacy
    for lang_code, model_name in SPACY_MODELS.items():
        try:
            spacy.util.get_package_path(model_name)
            print(f"  OK       {model_name}  ({lang_code})")
        except Exception:
            print(f"  MISSING  {model_name}  ({lang_code})")
            missing_models.append(model_name)
else:
    print("  spaCy itself is not installed, cannot check models.")

print()
print("=" * 70)
if not missing and not missing_models:
    print("Everything is installed. You can continue with Block 1.")
else:
    print("SOMETHING IS MISSING. Run the following in a terminal, then restart the kernel:")
    print()
    if missing:
        print("   pip install " + " ".join(missing))
    for model_name in missing_models:
        print(f"   python -m spacy download {model_name}")
    print()
    print("You only need the spaCy model for the language(s) you actually process.")
print("=" * 70)

PYTHON
3.11.9 at C:\Users\ciara\geoparser_old\Scripts\python.exe

REQUIRED PACKAGES
  OK       pandas
  OK       numpy
  OK       sklearn
  OK       scipy
  OK       joblib
  OK       tqdm
  OK       datasketch
  OK       spacy
  OK       geoparser
  OK       sentence_transformers
  OK       iterstrat
  OK       lxml

OPTIONAL PACKAGES (canton-level location check only)
  OK       geopandas
  OK       shapely

SPACY LANGUAGE MODELS
  OK       de_core_news_lg  (DE)
  OK       fr_core_news_lg  (FR)
  OK       it_core_news_lg  (IT)

Everything is installed. You can continue with Block 1.


---
# Block 1 — Settings

**This is the only cell you need to edit.** Everything below it reads these values.

The most common edits are the first three: which language, where your article file is,
and (if you process several languages) running the whole notebook again with a different
`LANGUAGE`.

In [1]:
# ============================================================================
# BLOCK 1 — SETTINGS
# ============================================================================

# ----------------------------------------------------------------------------
# 1. THE BASICS
# ----------------------------------------------------------------------------

# WHAT DO YOU WANT TO DO? One of "train" or "apply".
#
#   "train"  Learn the models from an article file that contains the annotated
#            articles (e.g. the 1990-2024 corpus). Runs the training blocks and
#            saves all three models to models/. Do this once.
#
#   "apply"  Take the models already trained and run them on NEW articles that
#            have have NOT been annotated (e.g. 2025 onwards). No annotations are needed
#            in the input. The training blocks are skipped and the models are loaded
#            from disk instead.
#
# Typical workflow: run once with MODE = "train" on the old corpus, then set
# MODE = "apply" and run again for each new batch of articles.
MODE = "apply"

# Which language are you processing? One of "DE", "FR", "IT".
# To process several languages, run the whole notebook once per language.
LANGUAGE = "FR"

# The raw article file (.tsv or .csv). The separator is detected automatically.
#   - in "train" mode: the corpus the annotations were made from
#   - in "apply" mode: the new, un-annotated articles to classify
INPUT_FILE = "input/FR_2025_S_secheresse_s.tsv"

# Folder holding the annotation CSVs.
ANNOTATION_DIR = "annotation"

# Folder where all results are written. Created automatically.
OUTPUT_DIR = "output_2025"


# ----------------------------------------------------------------------------
# 2. GEOPARSING  (both ON by default)
# ----------------------------------------------------------------------------
# Geoparsing recognises place names and looks them up in a gazetteer. It is the
# slowest part of the pipeline but also the part that makes the results Swiss-specific.
#
# Setting either of these to False makes the pipeline much faster but less accurate:
#   - Without the Swiss filter, articles about droughts abroad are NOT removed.
#   - Without location extraction, you get impact types but no places.

RUN_SWISS_FILTER = True        # Block 3: is this article about Switzerland?
RUN_LOCATION_EXTRACTION = True # Block 10: which Swiss places does it mention?


# ----------------------------------------------------------------------------
# 3. HOW MUCH TO RE-COMPUTE
# ----------------------------------------------------------------------------
# The geoparsing blocks save their progress as they go. If a run is interrupted,
# leave this as False and re-run the block: it picks up where it stopped.
# Set it to True only if you want to throw away previous results and start over.

FORCE_RECOMPUTE_GEOPARSING = False


# ----------------------------------------------------------------------------
# 4. ANNOTATION FILE PATTERNS
# ----------------------------------------------------------------------------
# The notebook collects EVERY file matching each pattern and merges them, so you can
# add new annotation batches at any time without touching this notebook.
# "*" means "anything can go here".
#
# Example: with the pattern below, all of these are found and merged automatically:
#     DE_annotated_impacts_locations.csv
#     DE_annotated_impacts_round2.csv
#     DE_annotated_impacts_2026.csv

ANNOTATION_PATTERNS = {
    "impacts":    "{lang}_annotated_impacts*.csv",     # span annotations (required)
    "irrelevant": "{lang}_annotated_irrelevant*.csv",  # article IDs, not about drought
    "relevant":   "{lang}_annotated_relevant*.csv",    # extra article IDs, about drought
    "foreign":    "{lang}_annotated_foreign*.csv",     # optional, Swiss-filter check only
}


# ----------------------------------------------------------------------------
# 5. FILTERING SETTINGS (Block 3)
# ----------------------------------------------------------------------------

# Near-duplicate detection: two articles count as duplicates when they share at least
# THRESHOLD of their 5-word sequences AND were published within MAX_DAYS of each other.
DEDUP_SHINGLE_SIZE = 5      # length of the word sequences compared
DEDUP_NUM_PERM     = 256    # accuracy of the similarity estimate (higher = slower)
DEDUP_THRESHOLD    = 0.8    # 0.8 = 80% overlap required
DEDUP_MAX_DAYS     = 7      # only compare articles published within a week
DEDUP_MAX_CHARS    = 1000   # compare only the first 1000 characters

# Geoparsing performance settings for the Swiss filter.
SWISS_FILTER_MAX_CHARS = 1000   # how much of each article to scan for place names
SWISS_FILTER_BATCH_SIZE = 16    # articles processed at once (lower = less memory)
SWISS_FILTER_SAVE_EVERY = 5     # save progress every N batches


# ----------------------------------------------------------------------------
# 6. MODEL SETTINGS
# ----------------------------------------------------------------------------

# --- Article relevance (Block 5) ---
# True  = train only on annotated articles that survived filtering (matches the original
#         notebooks). False = also use annotated articles that filtering removed.
ARTICLE_TRAIN_ON_FILTERED_ONLY = True

# --- Sentence relevance (Block 7) ---
MIN_SENTENCE_LENGTH = 15    # ignore sentences shorter than this many characters
TARGET_RECALL = 0.90        # find the threshold that catches 90% of true impact sentences
                            # while keeping precision as high as possible
SENTENCE_SEARCH = "full"    # "full" tries 216 model settings (slower, matches the
                            # original notebooks); "quick" tries 8 (for a first test run)

# --- Impact classification (Block 8) ---
EMBEDDER_NAME = "paraphrase-multilingual-mpnet-base-v2"  # multilingual sentence embeddings
IMPACT_CLASSIFIER = "logreg"   # "logreg", "svm" or "rf"
USE_EMBEDDINGS = True          # semantic features
USE_TFIDF = True               # word features
# Note: USE_POSITION and the split settings below are now set PER LANGUAGE, in
# LANGUAGE_SETTINGS further down, because they differ between DE, FR and IT.

# How much the embedding and word features count in the combined representation.
# (The position-feature weight is per language — see LANGUAGE_SETTINGS.)
W_EMBEDDINGS, W_TFIDF = 2.0, 2.0

# --- Synthetic (cross-lingual) training data for the impact classifier ---
# Optionally add machine-translated sentences from the OTHER languages to the training
# set. This helps a lot when a language has little annotated data of its own. Italian in
# particular is trained ENTIRELY on synthetic data (see LANGUAGE_SETTINGS below).
#
# Each synthetic file is a CSV with the same columns as the real training data:
#   sentence_raw, labels, file, sentence_relative_position
# The files to use are chosen per language in LANGUAGE_SETTINGS. This switch just turns
# the whole mechanism on or off.
USE_SYNTHETIC = True
SYNTHETIC_DIR = "classification"   # folder holding the synthetic CSVs

# Print a detailed table of how many examples of each label ended up in each split.
PRINT_SPLIT_DIAGNOSTICS = True
MIN_FILES_PER_LABEL_WARNING = 3    # warn if a label appears in fewer files than this

# --- Location clustering (Block 10) ---
CLUSTER_EPS_KM = 10        # places within 10 km are treated as the same location
MATCH_RADIUS_KM = 10       # a predicted location counts as correct within 10 km of the
                           # annotated one


# ----------------------------------------------------------------------------
# 7. PER-LANGUAGE DEFAULTS
# ----------------------------------------------------------------------------
# These were tuned per language in the original notebooks. Change them only if you
# know why you are changing them.
#
# The impact-classifier keys control how Block 8 trains:
#   impact_split_strategy   How train/val/test are formed:
#       "file_stratified"   Split articles into all three sets, keeping articles whole
#                           and balancing labels. Used for DE and FR.
#       "no_real_train"     Use NO real data for training; split the real data into
#                           validation + test only, and train purely on synthetic data.
#                           Used for IT.
#       "file_random"       Simple split by article, no label balancing.
#       "multilabel_sentence"  Balance at sentence level (allows the same article to
#                           appear in more than one split — usually not what you want).
#   impact_ratios           (train, val, test). Must sum to 1.0. For "no_real_train"
#                           the train share is 0.0 and the real data is split val/test.
#   impact_use_position     Whether to add the sentence-position feature.
#   impact_w_pos            Weight of that position feature (ignored if not used).
#   impact_synthetic_files  Which synthetic CSVs to add to the training data.

LANGUAGE_SETTINGS = {
    "DE": {
        "spacy_model": "de_core_news_lg",
        # Newspaper sections that are dropped outright (foreign news, lifestyle, ...)
        "rubric_blacklist": ["ausland", "international", "beauty"],
        "impact_random_state": 7,     # data-split seed for the impact classifier
        "salience_threshold": 0.7,    # how dominant a location cluster must be to be kept
        "impact_split_strategy": "file_stratified",
        "impact_ratios": (0.6, 0.2, 0.2),
        "impact_use_position": True,
        "impact_w_pos": 1.0,
        "impact_synthetic_files": ["DE_synthetic_from_FR.csv"],
    },
    "FR": {
        "spacy_model": "fr_core_news_lg",
        "rubric_blacklist": [
            "monde", "international", "australie", "états-unis", "etats-unis",
            "afghanistan", "californie", "france", "afrique", "amériques",
        ],
        "impact_random_state": 7,
        "salience_threshold": 0.6,
        "impact_split_strategy": "file_stratified",
        "impact_ratios": (0.4, 0.3, 0.3),
        "impact_use_position": True,
        "impact_w_pos": 1.0,
        "impact_synthetic_files": ["FR_synthetic_from_DE.csv"],
    },
    "IT": {
        "spacy_model": "it_core_news_lg",
        "rubric_blacklist": ["mondo"],
        "impact_random_state": 21,
        "salience_threshold": 0.7,
        # IT has little annotated data, so it is trained ENTIRELY on synthetic data.
        # The real IT data is used only for validation (threshold tuning) and testing.
        "impact_split_strategy": "no_real_train",
        "impact_ratios": (0.0, 0.5, 0.5),
        "impact_use_position": False,
        "impact_w_pos": 0.0,
        "impact_synthetic_files": ["IT_synthetic_from_DE.csv", "IT_synthetic_from_FR.csv"],
    },
}

# Fixed seed so that every run produces identical results.
RANDOM_STATE = 7

print(f"Settings loaded for language: {LANGUAGE}")
print(f"Input file:        {INPUT_FILE}")
print(f"Annotations from:  {ANNOTATION_DIR}")
print(f"Results go to:     {OUTPUT_DIR}")
print(f"Swiss filter:      {'ON' if RUN_SWISS_FILTER else 'OFF'}")
print(f"Location extraction: {'ON' if RUN_LOCATION_EXTRACTION else 'OFF'}")

Settings loaded for language: FR
Input file:        input/FR_2025_S_secheresse_s.tsv
Annotations from:  annotation
Results go to:     output_2025
Swiss filter:      ON
Location extraction: ON


---
# Block 2 — Setup and file check

This cell loads the shared tools, creates the output folder, and checks that the files pointed at in Block 1 actually exist. **Run it every time the notebook is opened**.

In [4]:
# ============================================================================
# BLOCK 2 — SETUP AND FILE CHECK
# ============================================================================

import ast
import contextlib
import glob
import os
import re
import sys
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# ----------------------------------------------------------------------------
# Check the mode and language, and pull the language settings
# ----------------------------------------------------------------------------
if MODE not in ("train", "apply"):
    raise ValueError(
        f"MODE is '{MODE}' but must be 'train' or 'apply'. Fix this in Block 1."
    )

# A convenience flag used throughout the notebook.
TRAINING = (MODE == "train")

if LANGUAGE not in LANGUAGE_SETTINGS:
    raise ValueError(
        f"LANGUAGE is '{LANGUAGE}' but must be one of {list(LANGUAGE_SETTINGS)}. "
        "Fix this in Block 1."
    )

print(f"MODE = '{MODE}'  ->  "
      + ("training the models from annotated data" if TRAINING
         else "applying already-trained models to new data"))

LANG_CFG = LANGUAGE_SETTINGS[LANGUAGE]
SPACY_MODEL = LANG_CFG["spacy_model"]
RUBRIC_BLACKLIST = LANG_CFG["rubric_blacklist"]
IMPACT_RANDOM_STATE = LANG_CFG["impact_random_state"]
SALIENCE_THRESHOLD = LANG_CFG["salience_threshold"]

# Per-language impact-classifier settings (used in Block 8).
IMPACT_SPLIT_STRATEGY = LANG_CFG["impact_split_strategy"]
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = LANG_CFG["impact_ratios"]
USE_POSITION = LANG_CFG["impact_use_position"]
W_POSITION = LANG_CFG["impact_w_pos"]
IMPACT_SYNTHETIC_FILES = LANG_CFG["impact_synthetic_files"]

# Sanity-check the ratios so a typo is caught here rather than deep inside Block 8.
if abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) > 1e-6:
    raise ValueError(
        f"impact_ratios for {LANGUAGE} are {LANG_CFG['impact_ratios']} and do not sum "
        "to 1.0. Fix them in LANGUAGE_SETTINGS (Block 1)."
    )
if IMPACT_SPLIT_STRATEGY == "no_real_train" and TRAIN_RATIO != 0.0:
    raise ValueError(
        f"{LANGUAGE} uses the 'no_real_train' strategy, so its train ratio must be 0.0, "
        f"but it is {TRAIN_RATIO}. Fix impact_ratios in Block 1."
    )

# ----------------------------------------------------------------------------
# Create output folders
# ----------------------------------------------------------------------------
MODEL_DIR = "models"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)


# ----------------------------------------------------------------------------
# Helpers for saving and loading the result of each block
# ----------------------------------------------------------------------------
# Every block writes one CSV and the next block reads it. That means you can stop
# after any block and continue later, and you can re-run a single block on its own.

def stage_path(name):
    """Full path of a pipeline file, e.g. 'output/DE_05_filtered.csv'."""
    return os.path.join(OUTPUT_DIR, f"{LANGUAGE}_{name}.csv")


def save_stage(df, name, quiet=False):
    """Save a dataframe safely (writes to a temporary file first, then renames)."""
    path = stage_path(name)
    tmp = path + ".tmp"
    df.to_csv(tmp, index=False, encoding="utf-8-sig")
    os.replace(tmp, path)
    if not quiet:
        print(f"  saved  {len(df):>8,} rows  ->  {path}")
    return path


def load_stage(name):
    """Load the file produced by an earlier block, with a helpful error if it is missing."""
    path = stage_path(name)
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"'{path}' does not exist yet.\n"
            f"You need to run the block that produces '{name}' before this one."
        )
    df = pd.read_csv(path, low_memory=False)
    print(f"  loaded {len(df):>8,} rows  <-  {path}")
    return df


def stage_exists(name):
    return os.path.exists(stage_path(name))


# ----------------------------------------------------------------------------
# Text cleaning helpers, shared by several blocks
# ----------------------------------------------------------------------------

def remove_html(text):
    """Strip HTML tags such as <p> or <br> from a text."""
    if pd.isna(text):
        return text
    return re.sub(r"<[^>]+>", " ", str(text)).strip()


def normalize_text(text):
    """Lowercase, drop punctuation and collapse whitespace. Used for duplicate detection."""
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_whitespace(s):
    """Collapse whitespace only. Used when comparing annotated spans to article text."""
    return re.sub(r"\s+", " ", str(s).strip()) if pd.notna(s) else ""


@contextlib.contextmanager
def suppress_output():
    """Hide the very chatty progress bars printed by the geoparser library."""
    with open(os.devnull, "w") as fnull:
        old_out, old_err = sys.stdout, sys.stderr
        sys.stdout, sys.stderr = fnull, fnull
        try:
            yield
        finally:
            sys.stdout, sys.stderr = old_out, old_err


def parse_label_list(value):
    """Read a list of labels back from a CSV cell, tolerating empty values."""
    if isinstance(value, list):
        return value
    if pd.isna(value) or str(value).strip() in ("", "[]"):
        return []
    try:
        return ast.literal_eval(str(value))
    except (ValueError, SyntaxError):
        return []


def merge_energy_labels(labels):
    """Treat energy_hydro, energy_nuclear etc. as one 'energy' category."""
    return sorted({("energy" if l.startswith("energy_") else l) for l in labels})


def build_features(embeddings, tfidf_matrix, positions):
    """Glue the enabled feature types into one matrix (used in Block 8)."""
    from scipy.sparse import csr_matrix, hstack

    blocks = []
    if USE_EMBEDDINGS:
        blocks.append(csr_matrix(embeddings) * W_EMBEDDINGS)
    if USE_TFIDF:
        blocks.append(tfidf_matrix * W_TFIDF)
    if USE_POSITION:
        blocks.append(csr_matrix(np.nan_to_num(positions, nan=0.0)) * W_POSITION)
    if not blocks:
        raise ValueError("Enable at least one of USE_EMBEDDINGS / USE_TFIDF / USE_POSITION.")
    return hstack(blocks)


def find_annotation_files(kind):
    """Return every annotation file matching one of the patterns from Block 1."""
    pattern = ANNOTATION_PATTERNS[kind].format(lang=LANGUAGE)
    return sorted(glob.glob(os.path.join(ANNOTATION_DIR, pattern)))


class StopExecution(Exception):
    """Halt a single notebook cell cleanly, without showing an error."""
    def _render_traceback_(self):
        return []


def skip_unless_training(block_name, next_block):
    """
    Used at the top of the training-only blocks. In APPLY mode it prints a short
    note and stops the current cell (nothing below it in the cell runs), so you can
    still run every cell top to bottom without hitting errors.
    """
    if not TRAINING:
        print(f"APPLY mode: {block_name} is a training step and is skipped.")
        print(f"Continue with {next_block}.")
        raise StopExecution


# ----------------------------------------------------------------------------
# Check that the input files exist
# ----------------------------------------------------------------------------
print("=" * 70)
print("FILE CHECK")
print("=" * 70)

problems = []

# --- the article file ---
if os.path.exists(INPUT_FILE):
    size_mb = os.path.getsize(INPUT_FILE) / 1_000_000
    print(f"  OK       article file        {INPUT_FILE}  ({size_mb:,.1f} MB)")
else:
    print(f"  MISSING  article file        {INPUT_FILE}")
    problems.append(
        f"The article file '{INPUT_FILE}' was not found. "
        "Check the INPUT_FILE setting in Block 1."
    )

# In TRAIN mode we need the annotation files. In APPLY mode we need the saved models
# instead, and annotations are not required at all.

# Names of the three model files a full "train" run produces.
ARTICLE_MODEL_PATH = os.path.join(MODEL_DIR, f"{LANGUAGE}_article_model.joblib")
SENTENCE_MODEL_PATH = os.path.join(MODEL_DIR, f"{LANGUAGE}_sentence_model.joblib")
IMPACT_MODEL_PATH = os.path.join(MODEL_DIR, f"{LANGUAGE}_impact_model.joblib")

if TRAINING:
    # --- the annotation folder (required for training) --------------------
    if not os.path.isdir(ANNOTATION_DIR):
        print(f"  MISSING  annotation folder   {ANNOTATION_DIR}/")
        problems.append(
            f"The annotation folder '{ANNOTATION_DIR}' was not found. "
            "Create it and put your annotation CSVs inside."
        )
    else:
        print(f"  OK       annotation folder   {ANNOTATION_DIR}/")

        for kind in ["impacts", "irrelevant", "relevant", "foreign"]:
            files = find_annotation_files(kind)
            pattern = ANNOTATION_PATTERNS[kind].format(lang=LANGUAGE)
            required = kind in ("impacts", "irrelevant")

            if files:
                print(f"           {kind:<11} {len(files)} file(s) matching {pattern}")
                for f in files:
                    print(f"                       - {os.path.basename(f)}")
            elif required:
                print(f"  MISSING  {kind:<11} no file matching {pattern}")
                problems.append(
                    f"No '{kind}' annotation file found (pattern: {pattern}). "
                    "This is required to train the models."
                )
            else:
                print(f"           {kind:<11} none (optional)")

    # --- synthetic training files for the impact classifier ---------------
    # For "no_real_train" languages (IT) these are required, because there is no real
    # training data. For the others they are optional but recommended.
    synthetic_required = (IMPACT_SPLIT_STRATEGY == "no_real_train")
    if USE_SYNTHETIC and IMPACT_SYNTHETIC_FILES:
        print(f"  synthetic  impact-classifier training files "
              f"({'required' if synthetic_required else 'optional'}):")
        for fname in IMPACT_SYNTHETIC_FILES:
            path = os.path.join(SYNTHETIC_DIR, fname)
            if os.path.exists(path):
                print(f"             OK       {path}")
            else:
                print(f"             MISSING  {path}")
                if synthetic_required:
                    problems.append(
                        f"Synthetic file '{path}' is missing. {LANGUAGE} is trained "
                        "entirely on synthetic data (no_real_train), so this file is "
                        "required. Put it in the SYNTHETIC_DIR folder, or change the "
                        "split strategy for this language in Block 1."
                    )
    elif synthetic_required:
        problems.append(
            f"{LANGUAGE} uses the 'no_real_train' strategy but USE_SYNTHETIC is False "
            "or no synthetic files are listed. It has no data to train on. Enable "
            "USE_SYNTHETIC and list its synthetic files in Block 1."
        )

else:
    # --- the saved models (required for applying) -------------------------
    print("  APPLY mode: annotations are not needed, checking for saved models...")
    for label, path in [
        ("article model", ARTICLE_MODEL_PATH),
        ("sentence model", SENTENCE_MODEL_PATH),
        ("impact model", IMPACT_MODEL_PATH),
    ]:
        if os.path.exists(path):
            print(f"  OK       {label:<14} {path}")
        else:
            print(f"  MISSING  {label:<14} {path}")
            problems.append(
                f"The {label} '{os.path.basename(path)}' was not found. "
                "Run the notebook once with MODE = 'train' first."
            )

print()
if problems:
    print("=" * 70)
    print("PROBLEMS FOUND — fix these before continuing:")
    print("=" * 70)
    for i, p in enumerate(problems, 1):
        print(f"  {i}. {p}")
else:
    print("=" * 70)
    print("All required files are present. You can continue with Block 3.")
    print("=" * 70)

MODE = 'apply'  ->  applying already-trained models to new data
FILE CHECK
  OK       article file        input/FR_2025_S_secheresse_s.tsv  (12.7 MB)
  APPLY mode: annotations are not needed, checking for saved models...
  OK       article model  models\FR_article_model.joblib
  OK       sentence model models\FR_sentence_model.joblib
  OK       impact model   models\FR_impact_model.joblib

All required files are present. You can continue with Block 3.


---
# Block 3 — Filtering

Raw news archives contain a lot of material that is not usable. This block removes it in
four passes, and records *why* each article was dropped rather than deleting it, so you
can always inspect the decisions afterwards.

| Pass | Removes | Column that records it |
|---|---|---|
| 3a | — (cleaning only) | — |
| 3b | Articles whose body text is character-for-character identical to an earlier one | `duplicate` = `exact_dup` |
| 3c | Articles that are ~80% identical to another published within a week (agency copy re-runs) | `duplicate` = `near_dup` |
| 3d | Articles from foreign-news or lifestyle sections | `rubric_relevance` = `irrelevant` |
| 3e | Articles that mention more foreign than Swiss places | `swiss` = `no` |

The column `keep_final` is `1` for articles that survived everything and `0` otherwise.

Run cells 3a–3f in order.

### 3a — Load and clean

Reads your article file, strips HTML, and creates lowercase copies of the text used for
duplicate detection.

In [2]:
# ============================================================================
# BLOCK 3a — LOAD AND CLEAN
# ============================================================================

print("Reading the article file...")

# Detect the separator: .tsv files are tab-separated, .csv files comma-separated.
separator = "\t" if INPUT_FILE.lower().endswith((".tsv", ".tab")) else ","
df = pd.read_csv(INPUT_FILE, sep=separator, low_memory=False)

print(f"  {len(df):,} articles read from {INPUT_FILE}")

# --- Check the columns we need are there ---------------------------------
REQUIRED_INPUT_COLUMNS = ["content_id", "pubtime", "content"]
missing_cols = [c for c in REQUIRED_INPUT_COLUMNS if c not in df.columns]
if missing_cols:
    raise ValueError(
        f"Your article file is missing these required columns: {missing_cols}\n"
        f"Columns found: {list(df.columns)}"
    )

# head / subhead / rubric are optional; create them empty if absent.
for col in ["head", "subhead", "rubric"]:
    if col not in df.columns:
        print(f"  note: no '{col}' column found, using empty values")
        df[col] = ""

# --- Clean the text -------------------------------------------------------
print("Removing HTML tags...")
for col in ["head", "subhead", "content"]:
    df[col] = df[col].apply(remove_html)

print("Creating normalised text for duplicate detection...")
df["head_lower"] = df["head"].apply(normalize_text)
df["subhead_lower"] = df["subhead"].apply(normalize_text)
df["content_lower"] = df["content"].apply(normalize_text)

# Length of the article, used later to break ties between duplicates.
df["char_count"] = df["content_lower"].str.len()

# --- Initialise the decision columns --------------------------------------
# Every article starts as "keep it" and is demoted by the passes that follow.
df["duplicate"] = "original"          # original / exact_dup / near_dup
df["rubric_relevance"] = "relevant"   # relevant / irrelevant
df["keep_final"] = 1                  # 1 = keep, 0 = drop

save_stage(df, "01_preprocessed")
print("\nDone. Continue with 3b.")

Reading the article file...


NameError: name 'pd' is not defined

### 3b — Exact duplicates

Newspapers re-publish the same agency text in several outlets. Where two articles have
literally identical body text, the earliest one is kept.

In [67]:
# ============================================================================
# BLOCK 3b — EXACT DUPLICATES
# ============================================================================

df = load_stage("01_preprocessed")

# Sort oldest first so that "keep the first occurrence" means "keep the original".
df["pubtime"] = pd.to_datetime(df["pubtime"], errors="coerce", utc=True)
df = df.sort_values("pubtime").reset_index(drop=True)

n_missing_dates = df["pubtime"].isna().sum()
if n_missing_dates:
    print(f"  warning: {n_missing_dates:,} articles have an unreadable publication date")

# Mark every repeat of an already-seen body text.
is_duplicate = df.duplicated(subset="content", keep="first")

df.loc[is_duplicate, "duplicate"] = "exact_dup"
df.loc[is_duplicate, "keep_final"] = 0

print(f"  exact duplicates found: {int(is_duplicate.sum()):,}")
print(f"  articles still in play: {int((df['keep_final'] == 1).sum()):,}")

save_stage(df, "02_dedup")
print("\nDone. Continue with 3c.")

  loaded    1,257 rows  <-  output_2025\FR_01_preprocessed.csv
  exact duplicates found: 175
  articles still in play: 1,082
  saved     1,257 rows  ->  output_2025\FR_02_dedup.csv

Done. Continue with 3c.


### 3c — Near duplicates

Lightly edited re-runs of the same agency story are not caught by an exact match. This
pass compares the first 1000 characters of each article using MinHash/LSH — a technique
that finds similar texts without comparing every pair, which would be far too slow.

Two articles are treated as versions of the same story when they share at least 80% of
their 5-word sequences **and** appeared within 7 days of each other. The earliest is
kept; if several share the earliest date, the longest is kept.

This cell takes a few minutes on a large corpus.

In [68]:
# ============================================================================
# BLOCK 3c — NEAR DUPLICATES
# ============================================================================

from datasketch import MinHash, MinHashLSH

df = load_stage("02_dedup")
df["pubtime"] = pd.to_datetime(df["pubtime"], errors="coerce")
df = df.reset_index(drop=True)


def build_minhash(text):
    """Turn a text into a compact 'fingerprint' that can be compared quickly."""
    text = text[:DEDUP_MAX_CHARS]
    tokens = text.split()

    # All overlapping sequences of DEDUP_SHINGLE_SIZE words.
    shingles = set(
        " ".join(tokens[i:i + DEDUP_SHINGLE_SIZE])
        for i in range(len(tokens) - DEDUP_SHINGLE_SIZE + 1)
    )

    m = MinHash(num_perm=DEDUP_NUM_PERM)
    for shingle in shingles:
        m.update(shingle.encode("utf-8"))
    return m


texts = df["content_lower"].fillna("").tolist()
dates = df["pubtime"].tolist()

print(f"Building fingerprints for {len(texts):,} articles...")
lsh = MinHashLSH(threshold=DEDUP_THRESHOLD, num_perm=DEDUP_NUM_PERM)
minhashes = {}

for i, text in enumerate(tqdm(texts, desc="fingerprints")):
    m = build_minhash(text)
    minhashes[i] = m
    lsh.insert(i, m)

print("Searching for near-duplicate groups...")
visited = set()
n_near_dups = 0

for i in tqdm(range(len(texts)), desc="comparing"):
    if i in visited:
        continue

    # Candidates that the index thinks are similar.
    candidates = [j for j in lsh.query(minhashes[i]) if j != i]
    if not candidates:
        continue

    # Keep only those published within the time window.
    base_date = dates[i]
    candidates = [
        j for j in candidates
        if pd.notna(base_date) and pd.notna(dates[j])
        and abs((dates[j] - base_date).days) <= DEDUP_MAX_DAYS
    ]

    # Confirm the similarity exactly (the index is fast but approximate).
    candidates = [
        j for j in candidates
        if minhashes[i].jaccard(minhashes[j]) >= DEDUP_THRESHOLD
    ]

    if not candidates:
        continue

    group = [i] + candidates
    group_df = df.loc[group]

    # Keep the earliest; if tied, keep the longest.
    earliest_date = group_df["pubtime"].min()
    earliest = group_df[group_df["pubtime"] == earliest_date]
    if len(earliest) == 1:
        keeper = earliest.index[0]
    else:
        keeper = earliest.sort_values("char_count", ascending=False).index[0]

    for idx in group:
        if idx != keeper and df.loc[idx, "duplicate"] == "original":
            df.loc[idx, "duplicate"] = "near_dup"
            df.loc[idx, "keep_final"] = 0
            n_near_dups += 1

    visited.update(group)

print(f"\n  near duplicates found:  {n_near_dups:,}")
print(f"  articles still in play: {int((df['keep_final'] == 1).sum()):,}")

save_stage(df, "02_dedup")
print("\nDone. Continue with 3d.")

  loaded    1,257 rows  <-  output_2025\FR_02_dedup.csv
Building fingerprints for 1,257 articles...


fingerprints:   0%|          | 0/1257 [00:00<?, ?it/s]

Searching for near-duplicate groups...


comparing:   0%|          | 0/1257 [00:00<?, ?it/s]


  near duplicates found:  49
  articles still in play: 1,033
  saved     1,257 rows  ->  output_2025\FR_02_dedup.csv

Done. Continue with 3d.


### 3d — Newspaper section filter

Articles filed under foreign-news or lifestyle sections are dropped, since a drought in
California is not a Swiss drought impact. The list of sections is `rubric_blacklist` in
Block 1 and differs per language.

In [69]:
# ============================================================================
# BLOCK 3d — NEWSPAPER SECTION FILTER
# ============================================================================

df = load_stage("02_dedup")

if "rubric" not in df.columns or df["rubric"].isna().all():
    print("  no usable 'rubric' column — skipping this filter")
else:
    print(f"  sections dropped for {LANGUAGE}: {RUBRIC_BLACKLIST}")

    # Build one pattern like "ausland|international|beauty" and match case-insensitively.
    pattern = "|".join(re.escape(word) for word in RUBRIC_BLACKLIST)
    matches_blacklist = df["rubric"].str.contains(pattern, case=False, na=False)

    df.loc[matches_blacklist, "rubric_relevance"] = "irrelevant"
    df.loc[matches_blacklist, "keep_final"] = 0

    print(f"  articles dropped:       {int(matches_blacklist.sum()):,}")
    print(f"  articles still in play: {int((df['keep_final'] == 1).sum()):,}")

save_stage(df, "03_rubric")
print("\nDone. Continue with 3e.")

  loaded    1,257 rows  <-  output_2025\FR_02_dedup.csv
  sections dropped for FR: ['monde', 'international', 'australie', 'états-unis', 'etats-unis', 'afghanistan', 'californie', 'france', 'afrique', 'amériques']
  articles dropped:       319
  articles still in play: 788
  saved     1,257 rows  ->  output_2025\FR_03_rubric.csv

Done. Continue with 3e.


### 3e — Swiss or foreign? (geoparsing)

This is the first of the two geoparsing steps, and one of the slowest parts of the
pipeline. It reads the first 1000 characters of each article, recognises the place names
in it, looks each one up in the GeoNames gazetteer, and compares how many Swiss places
are mentioned against how many foreign countries and continents.

- more Swiss places than foreign → `swiss = yes`
- more foreign than Swiss → `swiss = no` → **article dropped**
- none of either, or a tie → `swiss = unknown` → **article kept** (better to keep a
  doubtful article than to lose a real impact)

Progress is saved every few batches. If the run is interrupted, simply run this cell
again and it continues from where it stopped.

If `RUN_SWISS_FILTER = False` in Block 1, this cell marks everything `unknown` and moves
on without geoparsing.

In [6]:
# ============================================================================
# BLOCK 3e — SWISS OR FOREIGN? (GEOPARSING)
# ============================================================================

output_name = "04_geoparsed"

# Column names produced by this block
COL_SWISS = "swiss"                                  # yes / no / unknown
COL_SWISS_PLACES = "swiss_placenames"                # which Swiss places were found
COL_FOREIGN_PLACES = "foreign_country_placenames"    # which foreign countries were found
COL_FOREIGN_DOMINANT = "foreign_country_dominant"    # the most frequent foreign country

# ---------------------------------------------------------------------------
# Option A: geoparsing switched off
# ---------------------------------------------------------------------------
if not RUN_SWISS_FILTER:
    print("RUN_SWISS_FILTER is False — skipping geoparsing.")
    print("All articles will be marked 'unknown' and none will be dropped here.")

    df = load_stage("03_rubric")
    df[COL_SWISS] = "unknown"
    for col in [COL_SWISS_PLACES, COL_FOREIGN_PLACES, COL_FOREIGN_DOMINANT]:
        df[col] = None
    save_stage(df, output_name)

# ---------------------------------------------------------------------------
# Option B: geoparsing switched on
# ---------------------------------------------------------------------------
else:
    from geoparser import Geoparser

    TRANSFORMER_MODEL = "dguzh/geo-all-MiniLM-L6-v2"
    GAZETTEER = "geonames"
    SWISS_COUNTRY = "Switzerland"

    def classify_toponyms(doc):
        """Count the Swiss places and the foreign countries/continents in one document."""
        swiss_counter = Counter()
        foreign_counter = Counter()

        if not doc or not doc.toponyms:
            return swiss_counter, foreign_counter

        for toponym in doc.toponyms:
            location = toponym.location
            if not location:
                continue

            data = location.data if hasattr(location, "data") and location.data else location
            if not data:
                continue

            country = data.get("country_name")
            feature_type = data.get("feature_type")
            name = data.get("name") or toponym.text or "UNKNOWN"

            if not country:
                continue

            if country == SWISS_COUNTRY:
                # Any Swiss place counts: village, canton, mountain, river.
                swiss_counter[name] += 1
            elif feature_type in ("independent political entity", "continent"):
                # Only whole countries and continents count as "foreign", so that a
                # comparison with a German town does not make an article foreign.
                foreign_counter[name] += 1

        return swiss_counter, foreign_counter

    def decide_swiss(swiss_counter, foreign_counter):
        n_swiss = sum(swiss_counter.values())
        n_foreign = sum(foreign_counter.values())
        if n_swiss == 0 and n_foreign == 0:
            return "unknown"
        if n_swiss > n_foreign:
            return "yes"
        if n_foreign > n_swiss:
            return "no"
        return "unknown"

    # --- Resume from a previous run if there is one ------------------------
    if stage_exists(output_name) and not FORCE_RECOMPUTE_GEOPARSING:
        print("Found a previous geoparsing run — continuing from it.")
        df = load_stage(output_name)
    else:
        print("Starting geoparsing from scratch.")
        df = load_stage("03_rubric")

    for col in [COL_SWISS, COL_SWISS_PLACES, COL_FOREIGN_PLACES, COL_FOREIGN_DOMINANT]:
        if col not in df.columns:
            df[col] = None

    # Only process rows that do not have a valid answer yet.
    todo_mask = ~df[COL_SWISS].isin(["yes", "no", "unknown"])
    df_todo = df[todo_mask].copy()

    print(f"  articles already done:  {len(df) - len(df_todo):,}")
    print(f"  articles left to do:    {len(df_todo):,}")

    if df_todo.empty:
        print("  nothing left to process")
    else:
        print(f"\nLoading the geoparser ({SPACY_MODEL})... this takes a minute.")
        geoparser = Geoparser(
            spacy_model=SPACY_MODEL,
            transformer_model=TRANSFORMER_MODEL,
            gazetteer=GAZETTEER,
        )

        n_batches_done = 0
        batch_starts = range(0, len(df_todo), SWISS_FILTER_BATCH_SIZE)

        for start in tqdm(batch_starts, desc="geoparsing"):
            batch = df_todo.iloc[start:start + SWISS_FILTER_BATCH_SIZE]

            texts = (
                batch["head"].fillna("") + " " +
                batch["subhead"].fillna("") + " " +
                batch["content"].fillna("")
            ).astype(str).str[:SWISS_FILTER_MAX_CHARS].tolist()

            with suppress_output():
                docs = geoparser.parse(texts)

            flags, swiss_places, foreign_places, dominant = [], [], [], []

            for doc in docs:
                swiss_counter, foreign_counter = classify_toponyms(doc)
                flags.append(decide_swiss(swiss_counter, foreign_counter))
                swiss_places.append(str(dict(swiss_counter)))
                foreign_places.append(str(dict(foreign_counter)))
                dominant.append(
                    foreign_counter.most_common(1)[0][0] if foreign_counter else None
                )

            df.loc[batch.index, COL_SWISS] = flags
            df.loc[batch.index, COL_SWISS_PLACES] = swiss_places
            df.loc[batch.index, COL_FOREIGN_PLACES] = foreign_places
            df.loc[batch.index, COL_FOREIGN_DOMINANT] = dominant

            # Save progress regularly so an interruption costs little.
            n_batches_done += 1
            if n_batches_done % SWISS_FILTER_SAVE_EVERY == 0:
                save_stage(df, output_name, quiet=True)

    # Foreign articles are dropped.
    df.loc[df[COL_SWISS] == "no", "keep_final"] = 0

    print("\nResult:")
    print(df[COL_SWISS].value_counts(dropna=False).to_string())
    print(f"\n  articles still in play: {int((df['keep_final'] == 1).sum()):,}")

    save_stage(df, output_name)

print("\nDone. Continue with 3f.")

Found a previous geoparsing run — continuing from it.
  loaded    1,257 rows  <-  output_2025\FR_04_geoparsed.csv
  articles already done:  320
  articles left to do:    937

Loading the geoparser (fr_core_news_lg)... this takes a minute.


geoparsing:   0%|          | 0/59 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/128 [00:00<?, ?it/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

... (171 outputs left out)

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Batches:   0%|          | 0/30 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]


Result:
swiss
no         473
yes        426
unknown    358

  articles still in play: 590
  saved     1,257 rows  ->  output_2025\FR_04_geoparsed.csv

Done. Continue with 3f.


### 3f — The filtered dataset

Writes the articles that survived every filter, and prints a summary of what each pass
removed.

In [7]:
# ============================================================================
# BLOCK 3f — FILTERED DATASET AND FILTERING REPORT
# ============================================================================

df = load_stage("04_geoparsed")

filtered = df[df["keep_final"] == 1].copy()
save_stage(filtered, "05_filtered")

# ---------------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------------
print()
print("=" * 70)
print(f"FILTERING REPORT — {LANGUAGE}")
print("=" * 70)

total = len(df)
dup_counts = df["duplicate"].str.lower().value_counts()

print(f"\nArticles read in:                     {total:>10,}")
print(f"  removed as exact duplicates:        {dup_counts.get('exact_dup', 0):>10,}")
print(f"  removed as near duplicates:         {dup_counts.get('near_dup', 0):>10,}")

originals = df[df["duplicate"].str.lower() == "original"]
rubric_counts = originals["rubric_relevance"].str.lower().value_counts()
print(f"  removed by section filter:          {rubric_counts.get('irrelevant', 0):>10,}")

if RUN_SWISS_FILTER:
    still_ok = originals[originals["rubric_relevance"].str.lower() == "relevant"]
    swiss_counts = still_ok["swiss"].str.lower().value_counts()
    print(f"  removed as foreign:                 {swiss_counts.get('no', 0):>10,}")
    print()
    print("  Of the articles that reached the Swiss filter:")
    print(f"    clearly Swiss (yes):              {swiss_counts.get('yes', 0):>10,}")
    print(f"    undecidable (unknown, kept):      {swiss_counts.get('unknown', 0):>10,}")
    print(f"    foreign (no, dropped):            {swiss_counts.get('no', 0):>10,}")

print()
print(f"ARTICLES KEPT FOR ANALYSIS:           {len(filtered):>10,}")
print(f"  = {len(filtered) / total * 100:.1f}% of what you started with")
print("=" * 70)

  loaded    1,257 rows  <-  output_2025\FR_04_geoparsed.csv
  saved       590 rows  ->  output_2025\FR_05_filtered.csv

FILTERING REPORT — FR

Articles read in:                          1,257
  removed as exact duplicates:               175
  removed as near duplicates:                 49
  removed by section filter:                 245
  removed as foreign:                        198

  Of the articles that reached the Swiss filter:
    clearly Swiss (yes):                     353
    undecidable (unknown, kept):             237
    foreign (no, dropped):                   198

ARTICLES KEPT FOR ANALYSIS:                  590
  = 46.9% of what you started with


---
# Block 4 — Load the annotations

This block collects **every** annotation file matching the patterns from Block 1, merges
them, removes duplicate rows, and reports what it found.

This is the block that lets your training data grow. When you have annotated a new batch,
save it into the `annotation/` folder with a matching name (for example
`DE_annotated_impacts_round3.csv`) and re-run this block and everything after it. Nothing
else needs to change.

The block produces two things:

1. **Span annotations** — the highlighted passages with their impact labels, used in
   Blocks 6 to 8.
2. **Article labels** — a list of article IDs marked relevant or irrelevant, used in
   Block 5.

> **APPLY mode:** this block is skipped, because applying a trained model to new
> articles needs no annotations. Run it (and just run it — it does nothing) and move on.

In [8]:
# ============================================================================
# BLOCK 4 — LOAD ANNOTATIONS (ONE OR MANY FILES)
# ============================================================================

skip_unless_training("loading annotations", "Block 5")


def read_annotation_files(kind, required=False):
    """
    Read and merge every annotation file of one kind.

    Returns an empty dataframe when nothing is found and the kind is optional.
    A 'source_file' column records which file each row came from.
    """
    files = find_annotation_files(kind)

    if not files:
        pattern = ANNOTATION_PATTERNS[kind].format(lang=LANGUAGE)
        if required:
            raise FileNotFoundError(
                f"No '{kind}' annotation file found in '{ANNOTATION_DIR}' "
                f"(looking for: {pattern}).\n"
                "This file is required. Check ANNOTATION_DIR and ANNOTATION_PATTERNS "
                "in Block 1."
            )
        print(f"  {kind:<11} no files found (optional, skipping)")
        return pd.DataFrame()

    frames = []
    for path in files:
        one = pd.read_csv(path)
        one.columns = one.columns.str.strip()   # annotation exports often have stray spaces
        one["source_file"] = os.path.basename(path)
        frames.append(one)
        print(f"  {kind:<11} {len(one):>6,} rows from {os.path.basename(path)}")

    merged = pd.concat(frames, ignore_index=True)

    # Remove rows that appear in more than one file (e.g. a re-exported batch).
    before = len(merged)
    dedup_cols = [c for c in ["FILE", "ANCHOR", "TAGS", "QUOTE_TRANSCRIPTION"]
                  if c in merged.columns]
    if dedup_cols:
        merged = merged.drop_duplicates(subset=dedup_cols, keep="first")
    removed = before - len(merged)
    if removed:
        print(f"  {kind:<11} removed {removed:,} duplicate rows found in several files")

    return merged.reset_index(drop=True)


print("=" * 70)
print("READING ANNOTATION FILES")
print("=" * 70)

ann_impacts = read_annotation_files("impacts", required=True)
ann_irrelevant = read_annotation_files("irrelevant", required=True)
ann_relevant_extra = read_annotation_files("relevant", required=False)
ann_foreign = read_annotation_files("foreign", required=False)

# ---------------------------------------------------------------------------
# Check the span annotations have the columns we need
# ---------------------------------------------------------------------------
REQUIRED_ANNOTATION_COLUMNS = ["FILE", "ANCHOR", "TAGS"]
missing = [c for c in REQUIRED_ANNOTATION_COLUMNS if c not in ann_impacts.columns]
if missing:
    raise ValueError(
        f"Your span annotation files are missing these columns: {missing}\n"
        f"Columns found: {list(ann_impacts.columns)}"
    )

ann_impacts["FILE"] = ann_impacts["FILE"].astype(str).str.strip()

# Some exports have no COMMENTS column; create it so later code can rely on it.
if "COMMENTS" not in ann_impacts.columns:
    ann_impacts["COMMENTS"] = ""

save_stage(ann_impacts, "annotations_combined")

# ---------------------------------------------------------------------------
# Build the article-level labels used in Block 5
# ---------------------------------------------------------------------------
# An article is RELEVANT if it appears in the span annotations (it contains at least one
# annotated passage) or in an extra "relevant" list. Articles that annotators marked as
# "foreign" in the COMMENTS column are excluded, since they describe droughts abroad.

comments_lower = ann_impacts["COMMENTS"].fillna("").astype(str).str.lower().str.strip()
foreign_ids = set(ann_impacts.loc[comments_lower == "foreign", "FILE"])

relevant_ids = set(ann_impacts["FILE"]) - foreign_ids

if len(ann_relevant_extra) and "FILE" in ann_relevant_extra.columns:
    extra_ids = set(ann_relevant_extra["FILE"].astype(str).str.strip())
    print(f"\n  {len(extra_ids):,} extra relevant article IDs added")
    relevant_ids |= extra_ids

if "FILE" not in ann_irrelevant.columns:
    raise ValueError(
        "The 'irrelevant' annotation file needs a 'FILE' column with article IDs. "
        f"Columns found: {list(ann_irrelevant.columns)}"
    )
irrelevant_ids = set(ann_irrelevant["FILE"].astype(str).str.strip())

# An article cannot be both. If that happens, trust the span annotations.
conflicts = relevant_ids & irrelevant_ids
if conflicts:
    print(f"\n  warning: {len(conflicts):,} articles appear as BOTH relevant and irrelevant")
    print("           they are treated as relevant (span annotations win)")
    irrelevant_ids -= conflicts

article_labels = pd.concat([
    pd.DataFrame({"FILE": sorted(relevant_ids), "label": 1}),
    pd.DataFrame({"FILE": sorted(irrelevant_ids), "label": 0}),
], ignore_index=True)

save_stage(article_labels, "article_labels")

# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
print()
print("=" * 70)
print(f"ANNOTATION SUMMARY — {LANGUAGE}")
print("=" * 70)
print(f"  span annotations (rows):        {len(ann_impacts):>8,}")
print(f"  articles with span annotations: {ann_impacts['FILE'].nunique():>8,}")
print(f"  articles marked foreign:        {len(foreign_ids):>8,}")
print()
print(f"  articles labelled RELEVANT:     {len(relevant_ids):>8,}")
print(f"  articles labelled IRRELEVANT:   {len(irrelevant_ids):>8,}")

# Which impact labels were used, and how often?
if "TAGS" in ann_impacts.columns:
    tag_counts = (
        ann_impacts["TAGS"].dropna().astype(str)
        .str.split("|").explode().str.strip()
        .replace("", np.nan).dropna()
        .value_counts()
    )
    print()
    print("  Labels used in your annotations:")
    for tag, count in tag_counts.items():
        print(f"    {tag:<28} {count:>6,}")

    rare = tag_counts[tag_counts < 10]
    if len(rare):
        print()
        print("  warning: these labels have very few examples and will be hard to learn:")
        print("           " + ", ".join(rare.index))

print("=" * 70)

APPLY mode: loading annotations is a training step and is skipped.
Continue with Block 5.


StopExecution: 

---
# Block 5 — Article relevance

Surviving the filters in Block 3 only means an article mentions drought. It might still
be a wine review that praises a dry summer. This block tells articles that are genuinely
about drought from those that merely mention it.

It splits into two cells:

- **5a (train only)** learns the model from your annotated relevant/irrelevant articles,
  turns their text into word-frequency features (TF-IDF over single words and word pairs),
  compares several model types, trains a Linear SVC as the final model, and **saves it to
  disk**. Skipped in apply mode.
- **5b (both modes)** loads the saved model and adds an `is_relevant_article` column
  (`yes`/`no`) to every article in the current input.

So when you process new, un-annotated articles, 5a is skipped and 5b simply applies the
model you trained earlier.

In [9]:
# ============================================================================
# BLOCK 5a — TRAIN THE ARTICLE-RELEVANCE MODEL   (training mode only)
# ============================================================================

skip_unless_training("training the article-relevance model", "Block 5b")

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.svm import LinearSVC


def build_article_text(frame):
    """The model reads headline, sub-headline and body as one text."""
    return (
        frame["head"].fillna("") + " " +
        frame["subhead"].fillna("") + " " +
        frame["content"].fillna("")
    )


# ---------------------------------------------------------------------------
# 1. Assemble the training data
# ---------------------------------------------------------------------------
# By default we train on articles that survived filtering, which is what the original
# notebooks did. Set ARTICLE_TRAIN_ON_FILTERED_ONLY = False in Block 1 to also use
# annotated articles that filtering removed.

source_stage = "05_filtered" if ARTICLE_TRAIN_ON_FILTERED_ONLY else "04_geoparsed"
articles = load_stage(source_stage)
labels = load_stage("article_labels")

articles["content_id"] = articles["content_id"].astype(str)
labels["FILE"] = labels["FILE"].astype(str)

training = articles.merge(labels, left_on="content_id", right_on="FILE", how="inner")
training = training.dropna(subset=["label"])
training["label"] = training["label"].astype(int)
training["text"] = build_article_text(training)

n_lost = len(labels) - len(training)
if n_lost > 0:
    print(f"\n  note: {n_lost:,} annotated articles are not in '{source_stage}'")
    print("        (they were removed by filtering, or their IDs do not match this input)")

print(f"\n  usable labelled articles: {len(training):,}")
counts = training["label"].value_counts()
print(f"    relevant:   {counts.get(1, 0):,}")
print(f"    irrelevant: {counts.get(0, 0):,}")

if len(training) < 30:
    raise ValueError(
        f"Only {len(training)} labelled articles are available — far too few to train a "
        "model.\n"
        "The usual cause is that your INPUT_FILE does not contain the articles your "
        "annotations were made from. In TRAIN mode, point INPUT_FILE at the corpus you "
        "annotated (e.g. the 1990-2024 file). If you are trying to classify new, "
        "un-annotated articles, set MODE = 'apply' in Block 1 instead."
    )
if counts.get(0, 0) == 0 or counts.get(1, 0) == 0:
    raise ValueError(
        "You need both relevant AND irrelevant labelled articles to train this model."
    )

# ---------------------------------------------------------------------------
# 2. Split into training and test sets ('stratify' keeps the class ratio)
# ---------------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    training["text"], training["label"],
    test_size=0.2, random_state=RANDOM_STATE, stratify=training["label"],
)

tfidf = TfidfVectorizer(max_features=50_000, ngram_range=(1, 2))
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

print(f"\n  training on {len(X_train):,} articles, testing on {len(X_test):,}")

# ---------------------------------------------------------------------------
# 3. Compare candidate models (informative: shows the precision/recall trade-off)
# ---------------------------------------------------------------------------
print()
print("=" * 70)
print("MODEL COMPARISON (on the test set)")
print("=" * 70)
print(f"{'model':<22} {'irrelevant F1':>14} {'relevant F1':>13}")
print("-" * 70)

candidate_models = {
    "LogReg": LogisticRegression(max_iter=300),
    "LogReg_balanced": LogisticRegression(max_iter=300, class_weight="balanced"),
    "LinearSVC": LinearSVC(),
    "LinearSVC_balanced": LinearSVC(class_weight="balanced"),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    "ComplementNB": ComplementNB(),
    "MultinomialNB": MultinomialNB(),
}

for name, model in candidate_models.items():
    model.fit(X_train_vec, y_train)
    preds = model.predict(X_test_vec)
    _, _, f1, _ = precision_recall_fscore_support(
        y_test, preds, average=None, labels=[0, 1], zero_division=0
    )
    print(f"{name:<22} {f1[0]:>14.2f} {f1[1]:>13.2f}")

# ---------------------------------------------------------------------------
# 4. Train the final model
# ---------------------------------------------------------------------------
print()
print("=" * 70)
print("FINAL MODEL: Linear SVC")
print("=" * 70)

final_model = LinearSVC()
final_model.fit(X_train_vec, y_train)

test_preds = final_model.predict(X_test_vec)
print(classification_report(
    y_test, test_preds, target_names=["irrelevant", "relevant"], zero_division=0
))

# ---------------------------------------------------------------------------
# 5. Look at a few mistakes
# ---------------------------------------------------------------------------
test_articles = training.loc[X_test.index].copy()
test_articles["predicted"] = test_preds
false_positives = test_articles[(test_articles["label"] == 0) & (test_articles["predicted"] == 1)]
false_negatives = test_articles[(test_articles["label"] == 1) & (test_articles["predicted"] == 0)]


def show_errors(frame, title, max_rows=5):
    print(f"\n{title}: {len(frame)}")
    print("-" * 70)
    if frame.empty:
        print("  none")
        return
    for _, row in frame.head(max_rows).iterrows():
        preview = " ".join(str(row["content"]).split()[:25])
        print(f"  ID:   {row['content_id']}")
        print(f"  Head: {row['head']}")
        print(f"  Text: {preview}...")
        print("-" * 70)


show_errors(false_positives, "FALSE ALARMS (irrelevant, predicted relevant)")
show_errors(false_negatives, "MISSED (relevant, predicted irrelevant)")

# ---------------------------------------------------------------------------
# 6. Save the model and its vectorizer for later use (including apply mode)
# ---------------------------------------------------------------------------
import joblib

joblib.dump(
    {"model": final_model, "tfidf": tfidf},
    ARTICLE_MODEL_PATH,
)
print(f"\n  article-relevance model saved to {ARTICLE_MODEL_PATH}")

APPLY mode: training the article-relevance model is a training step and is skipped.
Continue with Block 5b.


StopExecution: 

### 5b — Apply the article-relevance model

Runs in **both** modes. It loads the saved model and labels every article in the current
input with `is_relevant_article`. In apply mode this is where your previously trained
model meets the new articles.

In [10]:
# ============================================================================
# BLOCK 5b — APPLY THE ARTICLE-RELEVANCE MODEL   (both modes)
# ============================================================================

import joblib

# Load the model (trained in 5a, or in an earlier training run).
saved = joblib.load(ARTICLE_MODEL_PATH)
article_model = saved["model"]
article_tfidf = saved["tfidf"]
print(f"  loaded article-relevance model from {ARTICLE_MODEL_PATH}")


def build_article_text(frame):
    return (
        frame["head"].fillna("") + " " +
        frame["subhead"].fillna("") + " " +
        frame["content"].fillna("")
    )


all_articles = load_stage("05_filtered")
all_articles = all_articles[all_articles["keep_final"] == 1].copy()
all_articles["text"] = build_article_text(all_articles)

X_all = article_tfidf.transform(all_articles["text"])
predictions = article_model.predict(X_all)

all_articles["is_relevant_article"] = pd.Series(
    predictions, index=all_articles.index
).map({0: "no", 1: "yes"})
# Distance from the decision boundary; values near zero are the uncertain cases.
all_articles["relevance_decision_score"] = article_model.decision_function(X_all)

print()
print("=" * 70)
print(f"ARTICLE RELEVANCE APPLIED — {LANGUAGE}  ({MODE} mode)")
print("=" * 70)
print(all_articles["is_relevant_article"].value_counts().to_string())
n_relevant = int((all_articles["is_relevant_article"] == "yes").sum())
print(f"\n  {n_relevant:,} articles go forward to sentence-level analysis")

save_stage(all_articles, "06_article_relevance")

  loaded article-relevance model from models\FR_article_model.joblib
  loaded      590 rows  <-  output_2025\FR_05_filtered.csv

ARTICLE RELEVANCE APPLIED — FR  (apply mode)
is_relevant_article
no     512
yes     78

  78 articles go forward to sentence-level analysis
  saved       590 rows  ->  output_2025\FR_06_article_relevance.csv


'output_2025\\FR_06_article_relevance.csv'

---
# Block 6 — Sentence preparation

Impacts are described in single sentences, not whole articles, so from here on the unit
of analysis is the sentence.

This block does two things:

1. **Splits** every relevant article into sentences.
2. **Matches** your annotated passages onto those sentences. Your annotation tool records
   each highlighted passage as a character range (the `ANCHOR` column, e.g.
   `...body[1]/div[3]::431` = "character 431 of the article body"). This block translates
   those ranges into sentence labels: a sentence that overlaps an annotated passage is a
   **relevant sentence**, one that does not is **irrelevant**.

Sentences from articles you never annotated are kept too, but left unlabelled — they are
what the finished model will be applied to.

The block prints a **match rate**. If it is well below 1.00, the character positions in
your annotations do not line up with the article text, usually because the text was
cleaned differently before annotation. Investigate before trusting the models.

In [11]:
# ============================================================================
# BLOCK 6 — SPLIT INTO SENTENCES AND MATCH ANNOTATED SPANS
# ============================================================================

import spacy

# Labels that describe the annotation rather than an impact, and are ignored.
IGNORE_LABELS = {"other", "general_statement"}

# The three text fields, in the order the annotation tool numbered them.
TEXT_COLUMNS = ["head", "subhead", "content"]

# ---------------------------------------------------------------------------
# 1. Load the articles that go forward
# ---------------------------------------------------------------------------
docs = load_stage("06_article_relevance")

# If the Swiss filter was switched off there is no 'swiss' column; treat all as unknown.
if "swiss" not in docs.columns:
    docs["swiss"] = "unknown"

before = len(docs)
docs = docs[
    (docs["duplicate"] == "original") &
    (docs["is_relevant_article"] == "yes") &
    (docs["swiss"].isin(["yes", "unknown"]))
].copy()
print(f"  {len(docs):,} of {before:,} articles go forward to sentence splitting")

for col in TEXT_COLUMNS:
    if col not in docs.columns:
        docs[col] = ""
    # Clean exactly as the annotated text was cleaned, so character positions line up.
    docs[col] = docs[col].apply(lambda t: "" if pd.isna(t) else re.sub(r"<[^>]+>", " ", str(t)).strip())

docs["content_id"] = docs["content_id"].astype(str)
docs = docs.drop_duplicates(subset=["content_id"], keep="first").set_index("content_id")

# ---------------------------------------------------------------------------
# 2. Read the annotated spans
# ---------------------------------------------------------------------------
# In apply mode there are no annotations: every sentence is simply left unlabelled,
# and the trained model (Block 7) decides which sentences are relevant.
if not TRAINING:
    print("  APPLY mode: no annotations to match; all sentences will be left unlabelled.")
    annotation_lookup = defaultdict(lambda: defaultdict(list))
    annotated_articles = set()
else:
    ann = load_stage("annotations_combined")

# Anchors look like:  from=/TEI/text[1]/body[1]/div[3]::431;to=/TEI/...::502
ANCHOR_PATTERN = re.compile(
    r"from=(?P<from_path>[^:]+)::(?P<start>\d+);to=(?P<to_path>[^:]+)::(?P<end>\d+)"
)


def parse_anchor(anchor):
    """Work out which text field a highlighted passage sits in, and where."""
    if pd.isna(anchor):
        return None

    match = ANCHOR_PATTERN.search(str(anchor))
    if not match:
        return None

    from_path = match.group("from_path")

    # div[1] = headline, div[2] = sub-headline, div[3] = body.
    if "teiHeader" in from_path and "titleStmt" in from_path and "title" in from_path:
        column = "head"
    elif "body[1]/div[1]" in from_path:
        column = "head"
    elif "body[1]/div[2]" in from_path:
        column = "subhead"
    elif "body[1]/div[3]" in from_path:
        column = "content"
    else:
        column = None

    return {
        "column": column,
        "span_start": int(match.group("start")),
        "span_end": int(match.group("end")),
    }


def parse_tags(tag_string):
    """Split 'agriculture|energy' into a list, dropping the labels we ignore."""
    if pd.isna(tag_string):
        return []
    return [t.strip() for t in str(tag_string).split("|")
            if t.strip() and t.strip() not in IGNORE_LABELS]


if TRAINING:
    # Keep only rows that carry a usable impact label.
    ann = ann[
        ann["TAGS"].notna() &
        (ann["TAGS"].astype(str).str.strip() != "") &
        (~ann["TAGS"].isin(IGNORE_LABELS))
    ].copy()

    ann["labels"] = ann["TAGS"].apply(parse_tags)
    parsed = ann["ANCHOR"].apply(parse_anchor)
    ann["column"] = parsed.apply(lambda x: x["column"] if isinstance(x, dict) else None)
    ann["span_start"] = parsed.apply(lambda x: x["span_start"] if isinstance(x, dict) else None)
    ann["span_end"] = parsed.apply(lambda x: x["span_end"] if isinstance(x, dict) else None)

    ann = ann[ann["labels"].map(bool) & ann["column"].notna()].copy()
    ann["FILE"] = ann["FILE"].astype(str)

    if "QUOTE_TRANSCRIPTION" not in ann.columns:
        ann["QUOTE_TRANSCRIPTION"] = ""
    if "UUID" not in ann.columns:
        ann["UUID"] = None

    print(f"  {len(ann):,} usable annotated passages")

    missing_docs = set(ann["FILE"]) - set(docs.index)
    if missing_docs:
        print(f"  warning: {len(missing_docs):,} annotated articles are not in the current")
        print("           article set (filtered out, or classified as irrelevant in Block 5)")
        print("           their annotations cannot be used for training")

    ann = ann[ann["FILE"].isin(docs.index)].copy()

    # -----------------------------------------------------------------------
    # 3. Check the character positions really point at the annotated text
    # -----------------------------------------------------------------------
    checks = []
    for _, row in ann.iterrows():
        text = docs.loc[row["FILE"], row["column"]]
        start, end = int(row["span_start"]), int(row["span_end"])

        extracted = None
        if isinstance(text, str) and 0 <= start <= end <= len(text):
            extracted = text[start:end]

        checks.append(
            normalize_whitespace(extracted) == normalize_whitespace(row["QUOTE_TRANSCRIPTION"])
        )

    match_rate = float(np.mean(checks)) if checks else 0.0
    print(f"\n  span match rate: {match_rate:.3f}   ({sum(checks):,} of {len(checks):,} exact)")
    if match_rate < 0.95:
        print("  WARNING: many annotated passages do not match the article text at the")
        print("           recorded position. Check that the text was cleaned the same way")
        print("           before annotation as it is here.")

    # Group the spans by article and text field for fast lookup.
    annotation_lookup = defaultdict(lambda: defaultdict(list))
    for _, row in ann.iterrows():
        annotation_lookup[row["FILE"]][row["column"]].append({
            "uuid": row["UUID"],
            "quote": row["QUOTE_TRANSCRIPTION"],
            "labels": row["labels"],
            "span_start": int(row["span_start"]),
            "span_end": int(row["span_end"]),
        })

    annotated_articles = set(annotation_lookup.keys())
    print(f"  {len(annotated_articles):,} articles carry annotations")

# ---------------------------------------------------------------------------
# 4. Split into sentences
# ---------------------------------------------------------------------------
# A blank spaCy pipeline with only a sentence splitter: fast, and enough for this job.
print(f"\nSplitting into sentences ({LANGUAGE})...")
nlp = spacy.blank(LANGUAGE.lower())
nlp.add_pipe("sentencizer")
nlp.max_length = 5_000_000

all_sentences = []
sentence_id = 0

for article_id, row in tqdm(docs.iterrows(), total=len(docs), desc="articles"):
    is_annotated = article_id in annotation_lookup
    position_in_article = 0

    for col in TEXT_COLUMNS:
        text = row.get(col, "")
        if not isinstance(text, str) or not text:
            continue

        spans_here = annotation_lookup.get(article_id, {}).get(col, [])

        for sent in nlp(text).sents:
            sent_text = sent.text
            if not sent_text.strip():
                continue

            sentence_id += 1
            position_in_article += 1
            sent_start, sent_end = sent.start_char, sent.end_char

            # Which annotated passages overlap this sentence?
            matched = [
                span for span in spans_here
                if not (span["span_end"] <= sent_start or span["span_start"] >= sent_end)
            ]

            matched_labels = sorted({
                label for span in matched for label in span["labels"]
            })

            all_sentences.append({
                "sentence_id": sentence_id,
                "FILE": article_id,
                "doc_sentence_id": position_in_article,
                "column": col,
                "sentence_raw": sent_text,
                "sentence_length": len(sent_text),
                "is_annotated_doc": int(is_annotated),
                # 1 / 0 for annotated articles, empty for the rest
                "is_relevant_sentence": int(len(matched) > 0) if is_annotated else None,
                "sent_start": sent_start,
                "sent_end": sent_end,
                "n_matched_spans": len(matched),
                "matched_spans_raw": [str(s["quote"]) for s in matched],
                "matched_labels": matched_labels,
            })

df_sentences = pd.DataFrame(all_sentences)

print()
print("=" * 70)
print("SENTENCE PREPARATION SUMMARY")
print("=" * 70)
print(f"  sentences created:              {len(df_sentences):>10,}")
print(f"  from annotated articles:        {int((df_sentences['is_annotated_doc'] == 1).sum()):>10,}")

labelled = df_sentences[df_sentences["is_annotated_doc"] == 1]
if len(labelled):
    print(f"    marked relevant:              {int((labelled['is_relevant_sentence'] == 1).sum()):>10,}")
    print(f"    marked irrelevant:            {int((labelled['is_relevant_sentence'] == 0).sum()):>10,}")
print(f"  awaiting classification:        {int((df_sentences['is_annotated_doc'] == 0).sum()):>10,}")
print("=" * 70)

save_stage(df_sentences, "07_sentences")

  loaded      590 rows  <-  output_2025\FR_06_article_relevance.csv
  78 of 590 articles go forward to sentence splitting
  APPLY mode: no annotations to match; all sentences will be left unlabelled.

Splitting into sentences (FR)...


articles:   0%|          | 0/78 [00:00<?, ?it/s]


SENTENCE PREPARATION SUMMARY
  sentences created:                   1,803
  from annotated articles:                 0
  awaiting classification:             1,803
  saved     1,803 rows  ->  output_2025\FR_07_sentences.csv


'output_2025\\FR_07_sentences.csv'

---
# Block 7 — Sentence relevance

Now the pipeline learns to recognise a sentence that reports a drought impact.

**Why a tuned threshold?** A classifier outputs a probability, and the usual cut-off of
0.5 is arbitrary. For this task missing a real impact is worse than passing on a doubtful
sentence, because the next block can still discard it. So instead of 0.5, the block finds
the cut-off that catches `TARGET_RECALL` (default 90%) of true impact sentences while
keeping precision as high as possible.

The search tries many combinations of settings and picks the best on a validation set
that the model never trained on. Articles are kept whole within each split, so sentences
from one article never appear in both training and test data — otherwise the scores would
be flattering and wrong.

`SENTENCE_SEARCH = "full"` in Block 1 tries 216 combinations and takes a while;
`"quick"` tries 8 and is fine for a first test run.

In [12]:
# ============================================================================
# BLOCK 7a — TRAIN THE SENTENCE RELEVANCE MODEL   (training mode only)
# ============================================================================

skip_unless_training("training the sentence-relevance model", "Block 7b")

import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, classification_report, f1_score,
    precision_recall_curve, precision_score, recall_score,
)
from sklearn.model_selection import ParameterGrid, train_test_split

df = load_stage("07_sentences")

# Train only on annotated articles, and ignore very short sentences (headers, captions).
df = df[df["is_annotated_doc"] == 1].copy()
df = df[df["sentence_length"] >= MIN_SENTENCE_LENGTH].copy()
df["sentence_raw"] = df["sentence_raw"].fillna("")
df["is_relevant_sentence"] = df["is_relevant_sentence"].astype(int)

print(f"  {len(df):,} annotated sentences available for training")
print(df["is_relevant_sentence"].value_counts().rename({0: "irrelevant", 1: "relevant"}).to_string())

if len(df) < 200:
    print("\n  warning: this is a small training set; the scores below will be unstable")

# ---------------------------------------------------------------------------
# Split by ARTICLE, not by sentence
# ---------------------------------------------------------------------------
article_labels = df.groupby("FILE")["is_relevant_sentence"].max().reset_index()

trainval_files, test_files = train_test_split(
    article_labels["FILE"], test_size=0.20, random_state=RANDOM_STATE,
    stratify=article_labels["is_relevant_sentence"],
)
trainval_labels = article_labels[article_labels["FILE"].isin(trainval_files)]

train_files, val_files = train_test_split(
    trainval_labels["FILE"], test_size=0.20, random_state=RANDOM_STATE,
    stratify=trainval_labels["is_relevant_sentence"],
)

train_df = df[df["FILE"].isin(train_files)]
val_df = df[df["FILE"].isin(val_files)]
test_df = df[df["FILE"].isin(test_files)]

X_train, y_train = train_df["sentence_raw"], train_df["is_relevant_sentence"].values
X_val, y_val = val_df["sentence_raw"], val_df["is_relevant_sentence"].values
X_test, y_test = test_df["sentence_raw"], test_df["is_relevant_sentence"].values

print(f"\n  train: {len(train_df):,} sentences from {train_df['FILE'].nunique():,} articles")
print(f"  val:   {len(val_df):,} sentences from {val_df['FILE'].nunique():,} articles")
print(f"  test:  {len(test_df):,} sentences from {test_df['FILE'].nunique():,} articles")


def choose_threshold(y_true, probabilities, target_recall):
    """
    Pick the probability cut-off with the best precision among those that reach the
    target recall. If no cut-off reaches it, fall back to the best F1.
    """
    precisions, recalls, thresholds = precision_recall_curve(y_true, probabilities)
    table = pd.DataFrame({
        "threshold": np.append(thresholds, 1.0),
        "precision": precisions,
        "recall": recalls,
    })

    good_enough = table[table["recall"] >= target_recall]
    if len(good_enough):
        best = good_enough.sort_values(
            ["precision", "threshold"], ascending=[False, False]
        ).iloc[0]
    else:
        table["f1"] = 2 * table["precision"] * table["recall"] / (
            table["precision"] + table["recall"] + 1e-9
        )
        best = table.sort_values("f1", ascending=False).iloc[0]

    return float(best["threshold"])


# ---------------------------------------------------------------------------
# Search for the best settings
# ---------------------------------------------------------------------------
if SENTENCE_SEARCH == "quick":
    param_grid = {
        "max_features": [40000], "ngram_range": [(1, 1), (1, 2)],
        "min_df": [2], "max_df": [0.95], "C": [1.0, 2.0],
        "class_weight": ["balanced", None], "solver": ["liblinear"],
    }
else:
    param_grid = {
        "max_features": [20000, 40000, 50000], "ngram_range": [(1, 1), (1, 2)],
        "min_df": [2, 3, 5], "max_df": [0.90, 0.95], "C": [0.5, 1.0, 2.0],
        "class_weight": ["balanced", None], "solver": ["liblinear"],
    }

configs = list(ParameterGrid(param_grid))
print(f"\nTrying {len(configs)} combinations of settings...")

results = []
best_score = -np.inf
best = None

for config in tqdm(configs, desc="search"):
    vectorizer = TfidfVectorizer(
        max_features=config["max_features"], ngram_range=config["ngram_range"],
        min_df=config["min_df"], max_df=config["max_df"],
        sublinear_tf=True, lowercase=True,
    )

    X_train_vec = vectorizer.fit_transform(X_train)
    X_val_vec = vectorizer.transform(X_val)
    X_test_vec = vectorizer.transform(X_test)

    model = LogisticRegression(
        C=config["C"], max_iter=2000,
        class_weight=config["class_weight"], solver=config["solver"],
    )
    model.fit(X_train_vec, y_train)

    val_probs = model.predict_proba(X_val_vec)[:, 1]
    val_ap = average_precision_score(y_val, val_probs)

    threshold = choose_threshold(y_val, val_probs, TARGET_RECALL)
    val_preds = (val_probs >= threshold).astype(int)

    val_precision = precision_score(y_val, val_preds, zero_division=0)
    val_recall = recall_score(y_val, val_preds, zero_division=0)

    # Among models that reach the recall target, prefer the most precise one.
    score = val_precision + 0.01 * val_ap

    results.append({
        **config, "val_ap": val_ap, "val_threshold": threshold,
        "val_precision": val_precision, "val_recall": val_recall,
        "val_f1": f1_score(y_val, val_preds, zero_division=0),
        "selection_score": score,
    })

    if score > best_score:
        best_score = score
        best = {"config": config, "model": model, "vectorizer": vectorizer,
                "threshold": threshold, "X_test_vec": X_test_vec}

results_df = pd.DataFrame(results).sort_values("selection_score", ascending=False)
results_df.to_csv(os.path.join(MODEL_DIR, f"{LANGUAGE}_sentence_model_search.csv"), index=False)

print("\nBest settings found:")
for key, value in best["config"].items():
    print(f"  {key}: {value}")
print(f"  chosen probability cut-off: {best['threshold']:.4f}")

# ---------------------------------------------------------------------------
# How well does it do on data it has never seen?
# ---------------------------------------------------------------------------
test_probs = best["model"].predict_proba(best["X_test_vec"])[:, 1]

print()
print("=" * 70)
print("TEST PERFORMANCE at the usual cut-off of 0.50")
print("=" * 70)
print(classification_report(y_test, (test_probs >= 0.5).astype(int),
                            target_names=["irrelevant", "relevant"], zero_division=0))

print("=" * 70)
print(f"TEST PERFORMANCE at the tuned cut-off of {best['threshold']:.4f}  <- the one used")
print("=" * 70)
print(classification_report(y_test, (test_probs >= best["threshold"]).astype(int),
                            target_names=["irrelevant", "relevant"], zero_division=0))

print(f"Average precision (threshold-free quality): {average_precision_score(y_test, test_probs):.4f}")

# ---------------------------------------------------------------------------
# Save the model for later use
# ---------------------------------------------------------------------------
joblib.dump(best["model"], os.path.join(MODEL_DIR, f"{LANGUAGE}_sentence_model.joblib"))
joblib.dump(best["vectorizer"], os.path.join(MODEL_DIR, f"{LANGUAGE}_sentence_tfidf.joblib"))
with open(os.path.join(MODEL_DIR, f"{LANGUAGE}_sentence_threshold.txt"), "w") as f:
    f.write(str(best["threshold"]))

print(f"\n  model saved to {MODEL_DIR}/")

APPLY mode: training the sentence-relevance model is a training step and is skipped.
Continue with Block 7b.


StopExecution: 

### 7b — Apply the model to every sentence

Runs the trained model over the whole corpus and adds two columns:

- `drought_relevance_prob_model` — the probability that a sentence describes an impact
- `drought_relevant_pred` — `1` or `0`, using the tuned cut-off

It also computes `sentence_relative_position`: where the sentence sits in its article, on
a scale from 0 (first sentence) to 1 (last). This is available as an optional feature for
the next block.

Note that the predictions here are the *model's*, including for articles you annotated —
the gold labels are deliberately **not** written over the predictions, so that Block 9 can
give an honest picture of pipeline performance.

In [13]:
# ============================================================================
# BLOCK 7b — APPLY THE SENTENCE MODEL TO THE WHOLE CORPUS
# ============================================================================

import joblib

model = joblib.load(os.path.join(MODEL_DIR, f"{LANGUAGE}_sentence_model.joblib"))
vectorizer = joblib.load(os.path.join(MODEL_DIR, f"{LANGUAGE}_sentence_tfidf.joblib"))
with open(os.path.join(MODEL_DIR, f"{LANGUAGE}_sentence_threshold.txt")) as f:
    threshold = float(f.read().strip())

df = load_stage("07_sentences")

df["drought_relevance_prob_model"] = np.nan
df["drought_relevant_pred"] = np.nan

# Only sentences long enough to be worth classifying.
long_enough = df["sentence_length"].fillna(0) >= MIN_SENTENCE_LENGTH
texts = df.loc[long_enough, "sentence_raw"].fillna("")

if len(texts):
    probabilities = model.predict_proba(vectorizer.transform(texts))[:, 1]
    df.loc[long_enough, "drought_relevance_prob_model"] = probabilities
    df.loc[long_enough, "drought_relevant_pred"] = (probabilities >= threshold).astype(int)

df["drought_relevant_pred"] = df["drought_relevant_pred"].fillna(0).astype("Int64")

# ---------------------------------------------------------------------------
# Where does each sentence sit in its article? (0 = start, 1 = end)
# ---------------------------------------------------------------------------
stats = df.groupby("FILE", as_index=False)["doc_sentence_id"].agg(
    max_doc_sentence_id="max", total_sentences="nunique"
)
df = df.merge(stats, on="FILE", how="left")

df["sentence_relative_position"] = df["doc_sentence_id"] / (df["total_sentences"] - 1)
df.loc[df["total_sentences"] == 1, "sentence_relative_position"] = 0
df = df.drop(columns=["total_sentences"])

print()
print("=" * 70)
print("SENTENCE CLASSIFICATION RESULT")
print("=" * 70)
n_relevant = int((df["drought_relevant_pred"] == 1).sum())
print(f"  sentences classified:           {int(long_enough.sum()):>10,}")
print(f"  predicted to describe an impact:{n_relevant:>10,}")
print(f"  articles containing at least one:{df[df['drought_relevant_pred'] == 1]['FILE'].nunique():>9,}")
print("=" * 70)

save_stage(df, "08_sentence_relevance")

  loaded    1,803 rows  <-  output_2025\FR_07_sentences.csv

SENTENCE CLASSIFICATION RESULT
  sentences classified:                1,777
  predicted to describe an impact:     1,169
  articles containing at least one:       78
  saved     1,803 rows  ->  output_2025\FR_08_sentence_relevance.csv


'output_2025\\FR_08_sentence_relevance.csv'

---
# Block 8 — Impact classification

The last modelling step: given a sentence that reports a drought impact, **which kind** of
impact is it? Agriculture, energy, drinking water, shipping, forest fire, and so on.

One sentence can report several impacts at once ("low water levels halted shipping and
forced the power station to throttle back"), so this is a *multi-label* problem: each
impact type gets its own yes/no decision rather than the sentence being forced into one
box.

**Features.** Two views of each sentence are combined:

- **Embeddings** — a multilingual neural model that places sentences with similar
  *meaning* near each other, so it can recognise a paraphrase it has never seen.
- **TF-IDF** — plain word counts, which reliably catch the domain vocabulary.

**Thresholds.** Each impact type gets its own cut-off, tuned on a validation set. Rare
labels need lower cut-offs than common ones, so a single shared threshold would quietly
suppress the rare types.

**Per-language training.** How the model is trained differs by language, and all of it is
controlled from `LANGUAGE_SETTINGS` in Block 1:

- **DE and FR** split their annotated articles into train / validation / test
  (`file_stratified`), keeping each article whole and balancing the impact types across
  the splits. Optionally, machine-translated ("synthetic") sentences from the other
  language are added to the training data to reinforce it.
- **IT** has too little annotated data to train on, so it uses `no_real_train`: the real
  Italian data is split into validation and test **only**, and the model is trained
  **entirely on synthetic** Italian sentences translated from DE and FR. The real data is
  still used to tune thresholds (validation) and to report honest scores (test).

Because of this, the synthetic files listed for a language must be present in
`SYNTHETIC_DIR` — required for IT, recommended for DE and FR. A diagnostics table shows
how many examples of each label landed in each split, so you can spot labels that are too
rare to learn.

Downloading the embedding model the first time takes a few minutes.

In [14]:
# ============================================================================
# BLOCK 8a — TRAIN THE IMPACT CLASSIFIER   (training mode only)
# ============================================================================

skip_unless_training("training the impact classifier", "Block 8b")

import ast
import joblib
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from scipy.sparse import csr_matrix, hstack
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.svm import LinearSVC
from sentence_transformers import SentenceTransformer


# ===========================================================================
# Split helpers — one per strategy. All split by ARTICLE so sentences from one
# article never straddle two splits (except "multilabel_sentence", which is
# offered for completeness but allows that leakage).
# ===========================================================================

def _file_label_table(frame):
    """One row per article, listing the union of impact labels seen in that article."""
    return (
        frame.groupby("file")["labels"]
        .apply(lambda rows: sorted({lab for row in rows for lab in row}))
        .reset_index()
    )


def split_file_stratified(frame):
    """Split articles into train/val/test, keeping labels balanced (DE, FR)."""
    table = _file_label_table(frame)
    binarizer = MultiLabelBinarizer()
    Y = binarizer.fit_transform(table["labels"])
    X = table["file"].values

    s1 = MultilabelStratifiedShuffleSplit(
        n_splits=1, test_size=(VAL_RATIO + TEST_RATIO), random_state=IMPACT_RANDOM_STATE
    )
    train_i, temp_i = next(s1.split(X, Y))

    s2 = MultilabelStratifiedShuffleSplit(
        n_splits=1, test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
        random_state=IMPACT_RANDOM_STATE,
    )
    val_i, test_i = next(s2.split(X[temp_i], Y[temp_i]))

    train_files, val_files, test_files = X[train_i], X[temp_i][val_i], X[temp_i][test_i]
    return _subset(frame, train_files), _subset(frame, val_files), _subset(frame, test_files)


def split_no_real_train(frame):
    """No real training data (IT): split the real data into val + test only."""
    table = _file_label_table(frame)
    binarizer = MultiLabelBinarizer()
    Y = binarizer.fit_transform(table["labels"])
    X = table["file"].values

    # Split the whole real dataset directly into validation and test.
    s = MultilabelStratifiedShuffleSplit(
        n_splits=1, test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
        random_state=IMPACT_RANDOM_STATE,
    )
    val_i, test_i = next(s.split(X, Y))
    val_files, test_files = X[val_i], X[test_i]

    empty_train = frame.iloc[0:0].reset_index(drop=True)  # real train is empty
    return empty_train, _subset(frame, val_files), _subset(frame, test_files)


def split_file_random(frame):
    """Simple split by article, no label balancing."""
    files = frame["file"].dropna().unique()
    train_files, temp_files = train_test_split(
        files, test_size=(VAL_RATIO + TEST_RATIO), random_state=IMPACT_RANDOM_STATE
    )
    val_files, test_files = train_test_split(
        temp_files, test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
        random_state=IMPACT_RANDOM_STATE,
    )
    return _subset(frame, train_files), _subset(frame, val_files), _subset(frame, test_files)


def _subset(frame, files):
    return frame[frame["file"].isin(files)].reset_index(drop=True)


def split_dataset(frame):
    """Dispatch to the split strategy chosen for this language in Block 1."""
    if IMPACT_SPLIT_STRATEGY == "file_stratified":
        return split_file_stratified(frame)
    if IMPACT_SPLIT_STRATEGY == "no_real_train":
        return split_no_real_train(frame)
    if IMPACT_SPLIT_STRATEGY == "file_random":
        return split_file_random(frame)
    raise ValueError(f"Unknown impact_split_strategy: {IMPACT_SPLIT_STRATEGY}")


def print_split_diagnostics(df_train, df_val, df_test, full):
    """Show how many examples of each label ended up in each split, and warn on gaps."""
    def sent_counts(frame):
        if frame.empty:
            return pd.Series(dtype=int)
        return frame["labels"].explode().value_counts()

    tr, va, te = sent_counts(df_train), sent_counts(df_val), sent_counts(df_test)
    all_labels = sorted(set(tr.index) | set(va.index) | set(te.index))

    print("\n  Sentence-level label support by split:")
    print(f"    {'label':<26} {'train':>7} {'val':>7} {'test':>7}")
    print("    " + "-" * 50)
    for lab in all_labels:
        print(f"    {lab:<26} {int(tr.get(lab, 0)):>7} "
              f"{int(va.get(lab, 0)):>7} {int(te.get(lab, 0)):>7}")

    # Warn about labels missing from a split where that matters.
    if not df_train.empty:
        missing_train = [l for l in all_labels if tr.get(l, 0) == 0]
        if missing_train:
            print(f"\n  warning: no TRAIN examples for: {', '.join(missing_train)}")
    for name, counts in [("VAL", va), ("TEST", te)]:
        missing = [l for l in all_labels if counts.get(l, 0) == 0]
        if missing:
            print(f"  warning: no {name} examples for: {', '.join(missing)} "
                  "(their scores will be meaningless)")


# ---------------------------------------------------------------------------
# 1. Real labelled data: annotated sentences that describe an impact
# ---------------------------------------------------------------------------
REQUIRED_IMPACT_COLUMNS = {"sentence_raw", "labels", "file", "sentence_relative_position"}

df = load_stage("08_sentence_relevance")
df = df[df["is_relevant_sentence"] == 1].copy()
df = df.rename(columns={"FILE": "file", "matched_labels": "labels"})
df["labels"] = df["labels"].apply(parse_label_list).apply(merge_energy_labels)
df = df[df["labels"].map(len) > 0].copy()
df = df.sample(frac=1, random_state=IMPACT_RANDOM_STATE).reset_index(drop=True)

print(f"  {len(df):,} real labelled impact sentences from {df['file'].nunique():,} articles")

label_counts = df["labels"].explode().value_counts()
print("\n  Impact types in the real data:")
for label, count in label_counts.items():
    marker = "   <- very few" if count < 20 else ""
    print(f"    {label:<28} {count:>6,}{marker}")

# Warn about labels that live in very few articles (hard to split reliably).
file_label_counts = _file_label_table(df)["labels"].explode().value_counts()
rare_labels = file_label_counts[file_label_counts < MIN_FILES_PER_LABEL_WARNING]
if len(rare_labels):
    print(f"\n  warning: these labels appear in fewer than {MIN_FILES_PER_LABEL_WARNING} "
          "articles and cannot be split reliably:")
    print("           " + ", ".join(rare_labels.index))

# ---------------------------------------------------------------------------
# 2. Split the REAL data (strategy chosen per language)
# ---------------------------------------------------------------------------
print(f"\n  split strategy for {LANGUAGE}: {IMPACT_SPLIT_STRATEGY}")
print(f"  ratios (train/val/test): {TRAIN_RATIO}/{VAL_RATIO}/{TEST_RATIO}")

df_train_real, df_val, df_test = split_dataset(df)
df_train_real = df_train_real.copy()
df_train_real["data_source"] = "real"

# ---------------------------------------------------------------------------
# 3. Add synthetic (cross-lingual) training sentences
# ---------------------------------------------------------------------------
# These reinforce the training data (DE, FR) or ARE the training data (IT).
train_parts = [df_train_real]

if USE_SYNTHETIC and IMPACT_SYNTHETIC_FILES:
    for fname in IMPACT_SYNTHETIC_FILES:
        path = os.path.join(SYNTHETIC_DIR, fname)
        synth = pd.read_csv(path)
        synth.columns = synth.columns.str.strip()

        missing = REQUIRED_IMPACT_COLUMNS - set(synth.columns)
        if missing:
            raise ValueError(
                f"Synthetic file '{path}' is missing columns: {missing}\n"
                f"It must have: {sorted(REQUIRED_IMPACT_COLUMNS)}"
            )

        # The 'labels' column in a CSV is text like "['agriculture', 'energy']".
        synth["labels"] = synth["labels"].apply(parse_label_list).apply(merge_energy_labels)
        synth = synth[synth["labels"].map(len) > 0].copy()
        synth["data_source"] = "synthetic"
        train_parts.append(synth)
        print(f"  added {len(synth):,} synthetic sentences from {fname}")

df_train = pd.concat(train_parts, ignore_index=True)

if df_train.empty:
    raise ValueError(
        "The training set is empty. For a 'no_real_train' language this means the "
        "synthetic files are missing or empty. Check SYNTHETIC_DIR and the file list "
        "in Block 1."
    )

# Make sure the position column exists on every part (synthetic files may lack it).
for part in (df_train, df_val, df_test):
    if "sentence_relative_position" not in part.columns:
        part["sentence_relative_position"] = 0.0
    part["sentence_relative_position"] = part["sentence_relative_position"].fillna(0.0)

print("\n  Final training set:")
print(df_train["data_source"].value_counts().to_string())
print(f"\n  train: {len(df_train):,} sentences "
      f"({df_train['file'].nunique():,} distinct article IDs)")
print(f"  val:   {len(df_val):,} sentences / {df_val['file'].nunique():,} articles (real)")
print(f"  test:  {len(df_test):,} sentences / {df_test['file'].nunique():,} articles (real)")

if PRINT_SPLIT_DIAGNOSTICS:
    print_split_diagnostics(df_train, df_val, df_test, df)

# ---------------------------------------------------------------------------
# 4. Turn sentences into numbers
# ---------------------------------------------------------------------------
if USE_EMBEDDINGS:
    print(f"\nLoading the embedding model ({EMBEDDER_NAME})...")
    print("(the first time, this downloads about 1 GB — later runs are fast)")
    embedder = SentenceTransformer(EMBEDDER_NAME)
else:
    print("\nUSE_EMBEDDINGS is False — working from word features alone.")
    embedder = None


def embed(texts):
    if not USE_EMBEDDINGS:
        return None
    return embedder.encode(
        [("" if pd.isna(t) else str(t)) for t in texts],
        normalize_embeddings=True, batch_size=32, show_progress_bar=False,
    )


tfidf = TfidfVectorizer(
    ngram_range=(1, 2), min_df=1, max_df=0.95,
    max_features=20_000, sublinear_tf=True, analyzer="word",
)

train_texts = df_train["sentence_raw"].tolist()
val_texts = df_val["sentence_raw"].tolist()
test_texts = df_test["sentence_raw"].tolist()

print("Encoding sentences...")
emb_train, emb_val, emb_test = embed(train_texts), embed(val_texts), embed(test_texts)

tfidf_train = tfidf.fit_transform(train_texts)
tfidf_val = tfidf.transform(val_texts)
tfidf_test = tfidf.transform(test_texts)

pos_train = df_train[["sentence_relative_position"]].astype(float).values
pos_val = df_val[["sentence_relative_position"]].astype(float).values
pos_test = df_test[["sentence_relative_position"]].astype(float).values

X_train = build_features(emb_train, tfidf_train, pos_train)
X_val = build_features(emb_val, tfidf_val, pos_val)
X_test = build_features(emb_test, tfidf_test, pos_test)

# Fit the label binarizer on all splits so a label seen only in val/test still has a
# column (important for IT, whose labels come from real val/test, not synthetic train).
mlb = MultiLabelBinarizer()
mlb.fit(pd.concat([df_train["labels"], df_val["labels"], df_test["labels"]]))
Y_train = mlb.transform(df_train["labels"])
Y_val = mlb.transform(df_val["labels"])
Y_test = mlb.transform(df_test["labels"])

print(f"\n  feature matrix: {X_train.shape[0]:,} sentences x {X_train.shape[1]:,} features")
print(f"  impact types:   {list(mlb.classes_)}")

# ---------------------------------------------------------------------------
# 5. Train
# ---------------------------------------------------------------------------
if IMPACT_CLASSIFIER == "logreg":
    classifier = OneVsRestClassifier(LogisticRegression(
        C=3.0, max_iter=1000, class_weight="balanced", solver="liblinear"))
elif IMPACT_CLASSIFIER == "svm":
    classifier = OneVsRestClassifier(CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", random_state=IMPACT_RANDOM_STATE), cv=3))
elif IMPACT_CLASSIFIER == "rf":
    classifier = OneVsRestClassifier(RandomForestClassifier(
        n_estimators=200, n_jobs=-1, class_weight="balanced_subsample",
        random_state=IMPACT_RANDOM_STATE))
else:
    raise ValueError(f"IMPACT_CLASSIFIER must be 'logreg', 'svm' or 'rf', not '{IMPACT_CLASSIFIER}'")

print(f"\nTraining the {IMPACT_CLASSIFIER} classifier on {len(df_train):,} sentences...")
classifier.fit(X_train, Y_train)

# ---------------------------------------------------------------------------
# 6. Tune one cut-off per impact type, on the REAL validation set
# ---------------------------------------------------------------------------
val_probs = classifier.predict_proba(X_val)
thresholds = {}
for i, label in enumerate(mlb.classes_):
    best_f1, best_t = 0.0, 0.5
    for t in np.linspace(0.05, 0.95, 91):
        f1 = f1_score(Y_val[:, i], (val_probs[:, i] >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    thresholds[label] = best_t

print("\n  Tuned cut-off per impact type (from the real validation set):")
for label, t in thresholds.items():
    print(f"    {label:<28} {t:.2f}")

# ---------------------------------------------------------------------------
# 7. Evaluate on the REAL held-out test set
# ---------------------------------------------------------------------------
test_probs = classifier.predict_proba(X_test)
Y_pred = np.column_stack([
    (test_probs[:, i] >= thresholds[label]).astype(int)
    for i, label in enumerate(mlb.classes_)
])

print()
print("=" * 70)
print(f"IMPACT CLASSIFIER PERFORMANCE (real test set, {LANGUAGE})")
print("=" * 70)
print(f"  Micro F1 (overall): {f1_score(Y_test, Y_pred, average='micro'):.4f}")
print(f"  Macro F1 (average across impact types): {f1_score(Y_test, Y_pred, average='macro'):.4f}")
print()
print(classification_report(Y_test, Y_pred, target_names=list(mlb.classes_), zero_division=0))

# ---------------------------------------------------------------------------
# 8. Save everything needed to reuse this model
# ---------------------------------------------------------------------------
artifacts = {
    "model": classifier, "mlb": mlb, "tfidf": tfidf,
    "embedder_name": EMBEDDER_NAME, "thresholds": thresholds,
    "config": {
        "LANGUAGE": LANGUAGE, "IMPACT_CLASSIFIER": IMPACT_CLASSIFIER,
        "USE_EMBEDDINGS": USE_EMBEDDINGS, "USE_TFIDF": USE_TFIDF,
        "USE_POSITION": USE_POSITION, "RANDOM_STATE": IMPACT_RANDOM_STATE,
        "split_strategy": IMPACT_SPLIT_STRATEGY,
        "synthetic_files": IMPACT_SYNTHETIC_FILES if USE_SYNTHETIC else [],
    },
}
joblib.dump(artifacts, IMPACT_MODEL_PATH)
print(f"\n  model saved to {IMPACT_MODEL_PATH}")

APPLY mode: training the impact classifier is a training step and is skipped.
Continue with Block 8b.


StopExecution: 

### 8b — Apply the impact classifier

Runs the classifier over every sentence that Block 7 flagged as describing an impact, and
writes the main result file of the whole pipeline.

Each row is one sentence, with a `pred_labels` column listing its impact types and a
`prob_<type>` column giving the confidence for each type.

In [15]:
# ============================================================================
# BLOCK 8b — APPLY THE IMPACT CLASSIFIER
# ============================================================================

import joblib
from sentence_transformers import SentenceTransformer

# If Block 8a was run in an earlier session, load the saved model instead of
# requiring you to retrain it.
if "classifier" not in globals():
    print("No model in memory — loading the saved one...")
    saved = joblib.load(os.path.join(MODEL_DIR, f"{LANGUAGE}_impact_model.joblib"))
    classifier = saved["model"]
    mlb = saved["mlb"]
    tfidf = saved["tfidf"]
    thresholds = saved["thresholds"]
    embedder = SentenceTransformer(saved["embedder_name"]) if USE_EMBEDDINGS else None

df_full = load_stage("08_sentence_relevance")
df_full = df_full.rename(columns={"FILE": "file"})

# Only sentences the previous block flagged as impact-bearing.
df_full = df_full[df_full["drought_relevant_pred"] == 1].copy()
print(f"  {len(df_full):,} sentences to classify")

if len(df_full) == 0:
    raise ValueError(
        "No sentences were flagged as drought-relevant, so there is nothing to classify. "
        "Check the results of Block 7."
    )

texts = df_full["sentence_raw"].fillna("").astype(str).tolist()

print("Encoding sentences...")
emb_full = embedder.encode(texts, normalize_embeddings=True, batch_size=32,
                           show_progress_bar=True) if USE_EMBEDDINGS else None
tfidf_full = tfidf.transform(texts) if USE_TFIDF else None
pos_full = df_full[["sentence_relative_position"]].astype(float).values

X_full = build_features(emb_full, tfidf_full, pos_full)

probabilities = classifier.predict_proba(X_full)
predictions = np.column_stack([
    (probabilities[:, i] >= thresholds[label]).astype(int)
    for i, label in enumerate(mlb.classes_)
])

df_full["pred_labels"] = [
    [mlb.classes_[i] for i in np.where(row == 1)[0]] for row in predictions
]
for i, label in enumerate(mlb.classes_):
    df_full[f"prob_{label}"] = probabilities[:, i]

print()
print("=" * 70)
print("IMPACT TYPES FOUND IN THE CORPUS")
print("=" * 70)
found = df_full["pred_labels"].explode().dropna().value_counts()
for label, count in found.items():
    print(f"  {label:<28} {count:>8,} sentences")

n_empty = int((df_full['pred_labels'].map(len) == 0).sum())
print(f"\n  sentences with no impact type assigned: {n_empty:,}")
print("  (these passed the relevance filter but matched no type confidently)")
print("=" * 70)

save_stage(df_full, "09_impact_predictions")

No model in memory — loading the saved one...
  loaded    1,803 rows  <-  output_2025\FR_08_sentence_relevance.csv
  1,169 sentences to classify
Encoding sentences...


Batches:   0%|          | 0/37 [00:00<?, ?it/s]


IMPACT TYPES FOUND IN THE CORPUS
  ecosystems_terrestrial            502 sentences
  ecosystems_aquatic                221 sentences
  publicwatersupply                  62 sentences
  energy                             62 sentences
  agriculture_general                44 sentences
  agriculture_livestock              40 sentences
  agriculture_annual                 27 sentences
  agriculture_permanent              25 sentences
  inlandnavigation                   21 sentences

  sentences with no impact type assigned: 270
  (these passed the relevance filter but matched no type confidently)
  saved     1,169 rows  ->  output_2025\FR_09_impact_predictions.csv


'output_2025\\FR_09_impact_predictions.csv'

---
# Block 9 — Pipeline evaluation

Blocks 5, 7 and 8 each reported how well they do on their own. This block asks the
question that actually matters: **how well does the whole chain work end to end?**

Errors compound. A sentence missed by Block 7 can never be classified by Block 8, no
matter how good Block 8 is. The evaluation therefore separates:

- **Binary performance** — did Block 7 find the impact sentences?
- **Clean performance** — how good is Block 8 when it is handed only true impact sentences?
- **End-to-end performance** — what the pipeline actually delivers, errors and all.
- **Recall decomposition** — where recall is being lost.
- **Hallucination rate** — when Block 7 passes on a sentence that is not really about
  impacts, which types does Block 8 wrongly assign to it?

All of it is measured on your annotated articles only, since those are the only ones where
the truth is known.

In [16]:
# ============================================================================
# BLOCK 9 — END-TO-END PIPELINE EVALUATION   (training mode only)
# ============================================================================
# Evaluation compares predictions against known labels. New articles have no labels,
# so there is nothing to evaluate against in apply mode — this block is skipped there.

skip_unless_training("pipeline evaluation", "Block 10")

from sklearn.metrics import (
    accuracy_score, average_precision_score, coverage_error, f1_score,
    hamming_loss, label_ranking_average_precision_score, precision_score,
    recall_score,
)
from sklearn.preprocessing import MultiLabelBinarizer

# ---------------------------------------------------------------------------
# Bring truth and prediction together
# ---------------------------------------------------------------------------
truth = load_stage("08_sentence_relevance").rename(
    columns={"FILE": "file", "matched_labels": "true_labels"}
)
predicted = load_stage("09_impact_predictions")

truth["true_labels"] = truth["true_labels"].apply(parse_label_list).apply(merge_energy_labels)
predicted["pred_labels"] = predicted["pred_labels"].apply(parse_label_list)

prob_columns = [c for c in predicted.columns if c.startswith("prob_")]

# Merge on sentence_id, which is unique. Merging on the sentence text would
# duplicate rows whenever the same sentence appears twice in one article.
df_eval = truth.merge(
    predicted[["sentence_id", "pred_labels"] + prob_columns],
    on="sentence_id", how="left",
)
df_eval["pred_labels"] = df_eval["pred_labels"].apply(
    lambda x: x if isinstance(x, list) else []
)
df_eval["drought_relevant_pred"] = df_eval["drought_relevant_pred"].fillna(0).astype(int)

# Only annotated articles have a known truth.
df_eval = df_eval[df_eval["is_annotated_doc"] == 1].copy()
print(f"  evaluating on {len(df_eval):,} annotated sentences")

if len(df_eval) == 0:
    raise ValueError("No annotated sentences available — cannot evaluate.")

impact_types = sorted({l for row in df_eval["true_labels"] for l in row})
mlb_eval = MultiLabelBinarizer(classes=impact_types)
Y_true = mlb_eval.fit_transform(df_eval["true_labels"])
Y_pred = mlb_eval.transform(df_eval["pred_labels"])

# ---------------------------------------------------------------------------
# 1. Did Block 7 find the impact sentences?
# ---------------------------------------------------------------------------
truly_relevant = df_eval["true_labels"].apply(lambda x: 1 if len(x) > 0 else 0)
binary_pred = df_eval["drought_relevant_pred"]

print()
print("=" * 70)
print("1. SENTENCE RELEVANCE (Block 7) — did it find the impact sentences?")
print("=" * 70)
binary_recall = recall_score(truly_relevant, binary_pred, zero_division=0)
print(f"  Precision: {precision_score(truly_relevant, binary_pred, zero_division=0):.4f}"
      "   (of the sentences it passed, how many really were impacts)")
print(f"  Recall:    {binary_recall:.4f}"
      "   (of the real impact sentences, how many it found)")
print(f"  F1:        {f1_score(truly_relevant, binary_pred, zero_division=0):.4f}")

# ---------------------------------------------------------------------------
# 2. How good is Block 8 on its own?
# ---------------------------------------------------------------------------
df_clean = df_eval[df_eval["true_labels"].apply(len) > 0]
Y_true_clean = mlb_eval.transform(df_clean["true_labels"])
Y_pred_clean = mlb_eval.transform(df_clean["pred_labels"])

print()
print("=" * 70)
print("2. IMPACT CLASSIFIER (Block 8) — measured on true impact sentences only")
print("=" * 70)
print(f"  Micro F1: {f1_score(Y_true_clean, Y_pred_clean, average='micro', zero_division=0):.4f}")
print(f"  Macro F1: {f1_score(Y_true_clean, Y_pred_clean, average='macro', zero_division=0):.4f}")
print()
print(f"  {'impact type':<28} {'precision':>10} {'recall':>10} {'F1':>10}")
print("  " + "-" * 60)
for i, label in enumerate(impact_types):
    p = precision_score(Y_true_clean[:, i], Y_pred_clean[:, i], zero_division=0)
    r = recall_score(Y_true_clean[:, i], Y_pred_clean[:, i], zero_division=0)
    f = f1_score(Y_true_clean[:, i], Y_pred_clean[:, i], zero_division=0)
    print(f"  {label:<28} {p:>10.3f} {r:>10.3f} {f:>10.3f}")

# ---------------------------------------------------------------------------
# 3. The whole pipeline
# ---------------------------------------------------------------------------
print()
print("=" * 70)
print("3. FULL PIPELINE — what you actually get")
print("=" * 70)
print(f"  Micro precision: {precision_score(Y_true, Y_pred, average='micro', zero_division=0):.4f}")
print(f"  Micro recall:    {recall_score(Y_true, Y_pred, average='micro', zero_division=0):.4f}")
print(f"  Micro F1:        {f1_score(Y_true, Y_pred, average='micro', zero_division=0):.4f}")
print(f"  Macro F1:        {f1_score(Y_true, Y_pred, average='macro', zero_division=0):.4f}")
print()
print(f"  Exact match rate: {accuracy_score(Y_true, Y_pred):.4f}"
      "  (sentences where every label is right)")
print(f"  Hamming loss:     {hamming_loss(Y_true, Y_pred):.4f}"
      "  (fraction of wrong label decisions; lower is better)")
print()
print(f"  {'impact type':<28} {'precision':>10} {'recall':>10} {'F1':>10}")
print("  " + "-" * 60)
for i, label in enumerate(impact_types):
    p = precision_score(Y_true[:, i], Y_pred[:, i], zero_division=0)
    r = recall_score(Y_true[:, i], Y_pred[:, i], zero_division=0)
    f = f1_score(Y_true[:, i], Y_pred[:, i], zero_division=0)
    print(f"  {label:<28} {p:>10.3f} {r:>10.3f} {f:>10.3f}")

# ---------------------------------------------------------------------------
# 4. Where is recall lost?
# ---------------------------------------------------------------------------
passed_and_relevant = (truly_relevant.values == 1) & (binary_pred.values == 1)
conditional_recall = recall_score(
    Y_true[passed_and_relevant], Y_pred[passed_and_relevant],
    average="micro", zero_division=0,
)
end_to_end_recall = recall_score(Y_true, Y_pred, average="micro", zero_division=0)

print()
print("=" * 70)
print("4. WHERE RECALL IS LOST")
print("=" * 70)
print(f"  Block 7 keeps:                       {binary_recall:.4f}")
print(f"  Block 8 keeps (of what reaches it):  {conditional_recall:.4f}")
print(f"  Expected end-to-end (the product):   {binary_recall * conditional_recall:.4f}")
print(f"  Actual end-to-end:                   {end_to_end_recall:.4f}")
print()
if binary_recall < conditional_recall:
    print("  -> Most recall is lost at the sentence-relevance step (Block 7).")
    print("     Lowering its threshold or annotating more data would help most.")
else:
    print("  -> Most recall is lost at the impact-classification step (Block 8).")
    print("     More examples of the weaker impact types would help most.")

# ---------------------------------------------------------------------------
# 5. Ranking quality, independent of any threshold
# ---------------------------------------------------------------------------
if prob_columns:
    scores = df_eval[prob_columns].fillna(0).values
    score_labels = [c.replace("prob_", "") for c in prob_columns]
    mlb_scores = MultiLabelBinarizer(classes=score_labels)
    Y_true_scores = mlb_scores.fit_transform(df_eval["true_labels"])

    print()
    print("=" * 70)
    print("5. RANKING QUALITY (independent of the cut-offs)")
    print("=" * 70)
    print(f"  Label ranking average precision: "
          f"{label_ranking_average_precision_score(Y_true_scores, scores):.4f}")
    print(f"  Coverage error:                  "
          f"{coverage_error(Y_true_scores, scores):.4f}")
    print()
    print("  Average precision per impact type:")
    aps = []
    for i, label in enumerate(score_labels):
        if Y_true_scores[:, i].sum() == 0:
            continue
        ap = average_precision_score(Y_true_scores[:, i], scores[:, i])
        aps.append(ap)
        print(f"    {label:<28} {ap:.4f}")
    if aps:
        print(f"    {'MEAN':<28} {np.mean(aps):.4f}")

# ---------------------------------------------------------------------------
# 6. What happens on sentences that should not have got through?
# ---------------------------------------------------------------------------
false_positives = df_eval[(binary_pred == 1) & (truly_relevant == 0)]

print()
print("=" * 70)
print("6. FALSE ALARMS — sentences Block 7 wrongly passed on")
print("=" * 70)
print(f"  count: {len(false_positives):,}")
if len(false_positives):
    Y_fp = mlb_eval.transform(false_positives["pred_labels"])
    print("\n  How often each impact type is invented on these sentences:")
    for i, label in enumerate(impact_types):
        print(f"    {label:<28} {Y_fp[:, i].mean():.3f}")
print("=" * 70)

APPLY mode: pipeline evaluation is a training step and is skipped.
Continue with Block 10.


StopExecution: 

---
# Block 10 — Location extraction

The second geoparsing step, and the one that turns the results into something mappable.
Block 3 only asked *is this article Swiss?*; this block asks *which Swiss places does it
talk about?*

It runs the geoparser again, this time restricted to Switzerland, over the headline,
sub-headline and body of every article that contains at least one impact sentence. Each
recognised place name is resolved to coordinates and a canton.

Then it works out which places the article is actually *about*. A long article may
mention a dozen villages in passing while reporting on one region. Two ideas do the work:

- **Clustering** — places within `CLUSTER_EPS_KM` (10 km) of each other are treated as one
  location, so "Thun", "Lake Thun" and a nearby hamlet collapse into a single place.
- **Salience** — each cluster scores on how often it is mentioned and how early it appears,
  since journalists name the important place in the opening lines. Clusters scoring at
  least `salience_threshold` of the top cluster are kept, so an article about two regions
  keeps both.

If `RUN_LOCATION_EXTRACTION = False` in Block 1, both cells skip themselves.

In [17]:
# ============================================================================
# BLOCK 10a — FIND SWISS PLACE NAMES
# ============================================================================

if not RUN_LOCATION_EXTRACTION:
    print("RUN_LOCATION_EXTRACTION is False — skipping location extraction.")
else:
    from geoparser import Geoparser

    LOCATION_BATCH_SIZE = 64
    TEXT_FIELDS = ["head", "subhead", "content"]

    # Which articles do we need locations for? Those with impact sentences.
    impacts = load_stage("09_impact_predictions")
    wanted_ids = set(impacts["file"].astype(str))
    print(f"  {len(wanted_ids):,} articles contain impact sentences")

    articles = load_stage("05_filtered").rename(columns={"content_id": "file"})
    articles["file"] = articles["file"].astype(str)
    articles = articles[articles["file"].isin(wanted_ids)].reset_index(drop=True)

    for col in TEXT_FIELDS:
        if col not in articles.columns:
            articles[col] = ""

    # --- Resume a previous run -------------------------------------------
    previous = pd.DataFrame()
    if stage_exists("10_locations_raw") and not FORCE_RECOMPUTE_GEOPARSING:
        previous = load_stage("10_locations_raw")
        done_ids = set(previous["file"].astype(str))
        articles = articles[~articles["file"].isin(done_ids)].reset_index(drop=True)
        print(f"  {len(done_ids):,} articles already done, {len(articles):,} to go")

    if articles.empty:
        print("  nothing left to process")
    else:
        print(f"\nLoading the geoparser ({SPACY_MODEL})...")
        geoparser = Geoparser(
            spacy_model=SPACY_MODEL,
            transformer_model="dguzh/geo-all-MiniLM-L6-v2",
            gazetteer="geonames",
        )

        # Only accept places that are in Switzerland.
        SWITZERLAND_ONLY = {"country_name": ["Switzerland"]}

        # One text per field, so we know where in the article each place was mentioned.
        texts, meta = [], []
        for _, row in articles.iterrows():
            for field in TEXT_FIELDS:
                text = str(row.get(field, "") or "")
                if text.strip():
                    texts.append(text)
                    meta.append({"file": row["file"], "pubtime": row.get("pubtime"),
                                 "field": field})

        print(f"  {len(texts):,} text sections to parse")

        results = []
        articles_with_places = set()

        for start in tqdm(range(0, len(texts), LOCATION_BATCH_SIZE), desc="geoparsing"):
            batch_texts = texts[start:start + LOCATION_BATCH_SIZE]
            batch_meta = meta[start:start + LOCATION_BATCH_SIZE]

            with suppress_output():
                parsed_docs = geoparser.parse(batch_texts, filter=SWITZERLAND_ONLY)

            for doc, info in zip(parsed_docs, batch_meta):
                for toponym in doc.toponyms:
                    location = toponym.location
                    if not location:
                        continue

                    articles_with_places.add(info["file"])
                    results.append({
                        "file": info["file"],
                        "pubtime": info["pubtime"],
                        "source_field": info["field"],
                        "toponym_text": toponym.text,       # as written in the article
                        "matched_name": location.get("name"),  # official gazetteer name
                        "canton": location.get("admin1_name"),
                        "latitude": location.get("latitude"),
                        "longitude": location.get("longitude"),
                        "geoname_id": location.get("geonameid"),
                        "feature_type": location.get("feature_type"),
                        "score": round(toponym.score, 3) if toponym.score else None,
                        "char_start": getattr(toponym, "start_char", None),
                        "char_end": getattr(toponym, "end_char", None),
                        "has_swiss_toponym": 1,
                    })

        geo_df = pd.DataFrame(results)

        # Record articles where no Swiss place was found, so they are not silently lost.
        without = set(articles["file"]) - articles_with_places
        if without:
            geo_df = pd.concat([
                geo_df,
                pd.DataFrame({"file": sorted(without), "has_swiss_toponym": 0}),
            ], ignore_index=True)

        if len(previous):
            geo_df = pd.concat([previous, geo_df], ignore_index=True)

        geo_df = geo_df.sort_values("file").reset_index(drop=True)

        print()
        print("=" * 70)
        print("PLACE NAMES FOUND")
        print("=" * 70)
        found = geo_df[geo_df["has_swiss_toponym"] == 1]
        print(f"  place mentions:                 {len(found):>10,}")
        print(f"  distinct places:                {found['matched_name'].nunique():>10,}")
        print(f"  articles with a Swiss place:    {found['file'].nunique():>10,}")
        print(f"  articles without one:           {len(without):>10,}")
        if "canton" in found.columns and found["canton"].notna().any():
            print("\n  Most mentioned cantons:")
            for canton, count in found["canton"].value_counts().head(10).items():
                print(f"    {str(canton):<28} {count:>8,}")
        print("=" * 70)

        save_stage(geo_df, "10_locations_raw")

  loaded    1,169 rows  <-  output_2025\FR_09_impact_predictions.csv
  78 articles contain impact sentences
  loaded      590 rows  <-  output_2025\FR_05_filtered.csv

Loading the geoparser (fr_core_news_lg)...
  234 text sections to parse


geoparsing:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/64 [00:00<?, ?it/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Batches:   0%|          | 0/27 [00:00<?, ?it/s]

Batches:   0%|          | 0/64 [00:00<?, ?it/s]

... (5 outputs left out)

Batches:   0%|          | 0/42 [00:00<?, ?it/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Batches:   0%|          | 0/27 [00:00<?, ?it/s]


PLACE NAMES FOUND
  place mentions:                        739
  distinct places:                       147
  articles with a Swiss place:            76
  articles without one:                    2

  Most mentioned cantons:
    Valais                            167
    Vaud                               62
    Grisons                            48
    Geneva                             41
    Neuchâtel                          39
    Bern                               27
    Fribourg                           25
    Jura                               20
    Ticino                             12
    Thurgau                            11
  saved       741 rows  ->  output_2025\FR_10_locations_raw.csv


### 10b — Which places is each article actually about?

Groups the raw place mentions into clusters and keeps the salient ones, producing one row
per **main location** per article — the file you would join to a map.

If your annotations contain coordinates (`LAT` / `LNG` columns), the cell also scores the
result against them: how often the predicted location lands within `MATCH_RADIUS_KM` of a
location a human picked out.

In [58]:
# ============================================================================
# BLOCK 10b — CLUSTER PLACE NAMES INTO MAIN LOCATIONS
# ============================================================================

if not RUN_LOCATION_EXTRACTION:
    print("RUN_LOCATION_EXTRACTION is False — skipping.")
else:
    from sklearn.cluster import DBSCAN
    from sklearn.metrics.pairwise import haversine_distances

    EARTH_RADIUS_KM = 6371.0088

    # Places that the gazetteer resolves badly or that are too vague to be useful.
    # Add your own problem cases here after inspecting the output.
    EXCLUDED_PLACES = {
        "Switzerland", "Schweiz", "Suisse", "Svizzera",
        "Europa", "Europe", "Erde", "Alp", "Alpes", "Canton", "Cantone", "Franken",
    }

    auto = load_stage("10_locations_raw")
    auto = auto[auto["has_swiss_toponym"] == 1].copy()

    auto["matched_name"] = auto["matched_name"].astype(str).str.strip()
    auto = auto[~auto["matched_name"].isin(EXCLUDED_PLACES)]
    auto = auto.dropna(subset=["latitude", "longitude", "char_start"])
    auto["file"] = auto["file"].astype(str).str.strip()

    print(f"  {len(auto):,} usable place mentions in {auto['file'].nunique():,} articles")

    # --- How early in the article was each place mentioned? ---------------
    # 1.0 = right at the start (probably the subject), 0.0 = at the very end.
    auto["max_char"] = auto.groupby("file")["char_start"].transform("max")
    auto["position_weight"] = np.where(
        auto["max_char"] > 0, 1 - (auto["char_start"] / auto["max_char"]), 1.0
    )

    # --- Group nearby places together -------------------------------------
    def cluster_one_article(group):
        radians = np.radians(group[["latitude", "longitude"]].values)
        clustering = DBSCAN(
            eps=CLUSTER_EPS_KM / EARTH_RADIUS_KM,  # DBSCAN wants radians
            min_samples=1,                          # every place joins some cluster
            metric="haversine",
        ).fit(radians)
        group = group.copy()
        group["cluster"] = clustering.labels_
        return group

    print("Clustering places by article...")
    clustered = pd.concat(
        [cluster_one_article(g) for _, g in auto.groupby("file")],
        ignore_index=True,
    )

    # --- Score each cluster ------------------------------------------------
    clusters = (
        clustered.groupby(["file", "cluster"])
        .agg(
            center_lat=("latitude", "mean"),
            center_lon=("longitude", "mean"),
            mentions=("matched_name", "count"),
            position_score=("position_weight", "sum"),
            place_names=("matched_name", lambda x: sorted(set(x))),
            canton=("canton", lambda x: x.mode().iloc[0] if len(x.mode()) else None),
        )
        .reset_index()
    )

    # Mentioned often AND mentioned early = the place the article is about.
    clusters["salience"] = clusters["mentions"] + 2.0 * clusters["position_score"]

    # Keep clusters close to the best one, so multi-region articles keep both.
    def keep_salient(group):
        return group[group["salience"] >= SALIENCE_THRESHOLD * group["salience"].max()]

    selected = pd.concat(
        [keep_salient(g) for _, g in clusters.groupby("file")], ignore_index=True
    )

    print()
    print("=" * 70)
    print("MAIN LOCATIONS PER ARTICLE")
    print("=" * 70)
    print(f"  salience threshold used ({LANGUAGE}): {SALIENCE_THRESHOLD}")
    print(f"  articles with a location:      {selected['file'].nunique():>10,}")
    print(f"  main locations kept:           {len(selected):>10,}")
    print(f"  average per article:           {len(selected) / selected['file'].nunique():>10.2f}")
    if selected["canton"].notna().any():
        print("\n  Coverage by canton:")
        for canton, count in selected["canton"].value_counts().head(15).items():
            print(f"    {str(canton):<28} {count:>8,}")
    print("=" * 70)

    save_stage(selected, "11_location_clusters")

    # -----------------------------------------------------------------------
    # Optional: score against annotated coordinates
    # -----------------------------------------------------------------------
    ann = load_stage("annotations_combined")

    has_coordinates = {"LAT", "LNG"}.issubset(ann.columns)
    if not has_coordinates:
        print("\n  (no LAT/LNG columns in your annotations, skipping the accuracy check)")
    else:
        manual = ann.copy()
        if "VOCAB_LABEL" in manual.columns:
            manual = manual[
                manual["VOCAB_LABEL"].notna() &
                (manual["VOCAB_LABEL"].str.lower() != "switzerland")
            ]
        manual = manual.rename(columns={"FILE": "file", "LAT": "manual_lat", "LNG": "manual_lon"})
        manual = manual[["file", "manual_lat", "manual_lon"]].dropna()
        manual["file"] = manual["file"].astype(str).str.strip()

        if manual.empty:
            print("\n  (no annotated coordinates found, skipping the accuracy check)")
        else:
            def distance_km(lat1, lon1, lat2, lon2):
                a = np.radians([[lat1, lon1]])
                b = np.radians([[lat2, lon2]])
                return haversine_distances(a, b)[0][0] * EARTH_RADIUS_KM

            annotated_ids = set(manual["file"]) & set(selected["file"])

            correct_articles = 0
            evaluated_articles = 0
            annotations_total = 0
            annotations_found = 0

            for file_id in annotated_ids:
                manual_rows = manual[manual["file"] == file_id]
                predicted_rows = selected[selected["file"] == file_id]
                if manual_rows.empty:
                    continue

                evaluated_articles += 1
                annotations_total += len(manual_rows)

                # Precision: does at least one prediction land near an annotation?
                for _, prediction in predicted_rows.iterrows():
                    distances = manual_rows.apply(
                        lambda m: distance_km(m["manual_lat"], m["manual_lon"],
                                              prediction["center_lat"],
                                              prediction["center_lon"]), axis=1)
                    if distances.min() <= MATCH_RADIUS_KM:
                        correct_articles += 1
                        break

                # Recall: was each annotated location found?
                for _, annotation in manual_rows.iterrows():
                    if predicted_rows.empty:
                        continue
                    distances = predicted_rows.apply(
                        lambda c: distance_km(annotation["manual_lat"],
                                              annotation["manual_lon"],
                                              c["center_lat"], c["center_lon"]), axis=1)
                    if distances.min() <= MATCH_RADIUS_KM:
                        annotations_found += 1

            if evaluated_articles:
                precision = correct_articles / evaluated_articles
                recall = annotations_found / annotations_total if annotations_total else 0
                f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0

                print()
                print("=" * 70)
                print(f"LOCATION ACCURACY — checked against {evaluated_articles:,} annotated articles")
                print(f"(a prediction counts as correct within {MATCH_RADIUS_KM} km)")
                print("=" * 70)
                print(f"  Precision: {precision:.3f}   (articles where a predicted location was right)")
                print(f"  Recall:    {recall:.3f}   (annotated locations the pipeline found)")
                print(f"  F1:        {f1:.3f}")
                print("=" * 70)

  loaded       72 rows  <-  output_2025\IT_10_locations_raw.csv
  39 usable place mentions in 5 articles
Clustering places by article...

MAIN LOCATIONS PER ARTICLE
  salience threshold used (IT): 0.7
  articles with a location:               5
  main locations kept:                    9
  average per article:                 1.80

  Coverage by canton:
    Ticino                              2
    Valais                              2
    Thurgau                             1
    Grisons                             1
    Saint Gallen                        1
    Aargau                              1
  saved         9 rows  ->  output_2025\IT_11_location_clusters.csv


FileNotFoundError: 'output_2025\IT_annotations_combined.csv' does not exist yet.
You need to run the block that produces 'annotations_combined' before this one.

---
# Block 11 — Summary

Lists everything the pipeline produced and where to find it. Run this at the end, or any
time you want to see how far you got.

In [59]:
# ============================================================================
# BLOCK 11 — WHAT WAS PRODUCED
# ============================================================================

DESCRIPTIONS = [
    ("01_preprocessed",       "all articles, cleaned"),
    ("02_dedup",              "duplicates marked"),
    ("03_rubric",             "newspaper section filter applied"),
    ("04_geoparsed",          "Swiss/foreign decision added"),
    ("05_filtered",           "articles that survived all filters"),
    ("annotations_combined",  "all your annotation files merged"),
    ("article_labels",        "relevant/irrelevant article list"),
    ("06_article_relevance",  "articles classified as about drought or not"),
    ("07_sentences",          "sentences with annotated spans matched onto them"),
    ("08_sentence_relevance", "sentences scored for drought relevance"),
    ("09_impact_predictions", "MAIN RESULT: sentences with impact types"),
    ("10_locations_raw",      "every Swiss place name found"),
    ("11_location_clusters",  "MAIN RESULT: main location(s) per article"),
]

print("=" * 78)
print(f"PIPELINE OUTPUT — {LANGUAGE}")
print("=" * 78)
print(f"{'file':<34} {'rows':>10}   description")
print("-" * 78)

for name, description in DESCRIPTIONS:
    path = stage_path(name)
    if os.path.exists(path):
        rows = sum(1 for _ in open(path, encoding="utf-8-sig")) - 1
        print(f"{os.path.basename(path):<34} {rows:>10,}   {description}")
    else:
        print(f"{os.path.basename(path):<34} {'-':>10}   not produced yet")

print("-" * 78)
print(f"\nAll files are in:  {os.path.abspath(OUTPUT_DIR)}")
print(f"Trained models in: {os.path.abspath(MODEL_DIR)}")

print()
print("=" * 78)
print("WHAT TO DO WITH THE RESULTS")
print("=" * 78)
print(f"""
  {LANGUAGE}_09_impact_predictions.csv
      One row per impact sentence. Key columns:
        file                       article ID
        sentence_raw               the sentence itself
        pred_labels                impact types assigned to it
        prob_<type>                confidence for each type
        drought_relevance_prob_model   how impact-like the sentence is

  {LANGUAGE}_11_location_clusters.csv
      One row per main location per article. Key columns:
        file                       article ID
        center_lat / center_lon    coordinates
        canton                     canton it falls in
        place_names                the place names in that cluster
        salience                   how central it is to the article

  Join the two on 'file' to get impacts with places, then group by date to get
  a time series, or by canton to get a map.
""")
print("=" * 78)

PIPELINE OUTPUT — IT
file                                     rows   description
------------------------------------------------------------------------------
IT_01_preprocessed.csv                    111   all articles, cleaned
IT_02_dedup.csv                           111   duplicates marked
IT_03_rubric.csv                          111   newspaper section filter applied
IT_04_geoparsed.csv                       111   Swiss/foreign decision added
IT_05_filtered.csv                         62   articles that survived all filters
IT_annotations_combined.csv                 -   not produced yet
IT_article_labels.csv                       -   not produced yet
IT_06_article_relevance.csv                62   articles classified as about drought or not
IT_07_sentences.csv                       178   sentences with annotated spans matched onto them
IT_08_sentence_relevance.csv              178   sentences scored for drought relevance
IT_09_impact_predictions.csv              163   MAIN RESU

---
---
# Appendix A — Prepare a new batch of articles for annotation

**Not part of a normal run.** Use this when you want to grow your training data.

The workflow is:

1. Run this cell. It picks articles you have **not** annotated yet and writes them as TEI
   XML files, one per article.
2. Import those files into your annotation tool and annotate them.
3. Export the annotations as CSV and save the file into your `annotation/` folder with a
   matching name, e.g. `DE_annotated_impacts_round3.csv`.
4. Re-run Block 4 onwards. The new file is picked up automatically and merged with the
   existing ones — no code changes needed.

Annotating another 50–100 articles is almost always the most effective way to improve
results, far more so than changing model settings.

In [ ]:
# ============================================================================
# APPENDIX A — EXPORT A NEW SAMPLE FOR ANNOTATION
# ============================================================================

from pathlib import Path
import random

from lxml import etree

# --- Settings for this export ----------------------------------------------
N_ARTICLES = 20                  # how many articles to sample
EXCLUDE_YEARS = []               # e.g. ["2003", "2018"] to avoid over-covered years
SWISS_VALUES = ["yes", "unknown"]  # which Swiss-filter outcomes to draw from
FORCE_ARTICLE_IDS = []           # specific article IDs you want included
KEYWORDS = []                    # e.g. ["ernte", "waldbrand"]: forces one article per keyword
MAX_KEYWORD_CHARS = 100          # only search the opening of the article for keywords

SAMPLE_CSV = os.path.join(OUTPUT_DIR, f"{LANGUAGE}_annotation_sample.csv")
TEI_DIR = os.path.join(OUTPUT_DIR, f"{LANGUAGE}_annotation_sample_tei")

random.seed(RANDOM_STATE)

# --- Pool: articles that survived filtering and are not yet annotated -------
pool = load_stage("04_geoparsed")
already_annotated = set(load_stage("annotations_combined")["FILE"].astype(str))

pool["content_id"] = pool["content_id"].astype(str)
pool["year"] = pool["pubtime"].astype(str).str[:4]

if "swiss" not in pool.columns:
    pool["swiss"] = "unknown"

pool = pool[
    (pool["duplicate"] == "original") &
    (~pool["content_id"].isin(already_annotated)) &
    (~pool["year"].isin(EXCLUDE_YEARS)) &
    (pool["swiss"].isin(SWISS_VALUES))
].reset_index(drop=True)

print(f"  {len(pool):,} articles available to sample from")
print(f"  ({len(already_annotated):,} are already annotated and were excluded)")

if len(pool) < N_ARTICLES:
    raise ValueError(f"Only {len(pool)} articles available, but {N_ARTICLES} requested.")

# --- Choose the articles ----------------------------------------------------
chosen = set()

if FORCE_ARTICLE_IDS:
    forced = pool[pool["content_id"].isin([str(i) for i in FORCE_ARTICLE_IDS])]
    chosen.update(forced.index)
    print(f"  {len(forced)} of {len(FORCE_ARTICLE_IDS)} requested IDs found")

if KEYWORDS:
    search_text = (
        pool["head"].fillna("") + " " + pool["subhead"].fillna("") + " " +
        pool["content"].fillna("")
    ).str.lower()

    for keyword in KEYWORDS:
        matches = pool[search_text.str[:MAX_KEYWORD_CHARS].str.contains(
            re.escape(keyword.lower()), na=False)]
        available = [i for i in matches.index if i not in chosen]
        if available:
            chosen.add(random.choice(available))
            print(f"  keyword '{keyword}': 1 article selected")
        else:
            print(f"  keyword '{keyword}': no new matches")

remaining_needed = N_ARTICLES - len(chosen)
if remaining_needed < 0:
    raise ValueError(f"N_ARTICLES ({N_ARTICLES}) is smaller than the {len(chosen)} forced picks.")

remaining_pool = [i for i in pool.index if i not in chosen]
chosen.update(random.sample(remaining_pool, remaining_needed))

sample = pool.loc[sorted(chosen)].sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
sample.to_csv(SAMPLE_CSV, index=False)
print(f"\n  {len(sample)} articles sampled -> {SAMPLE_CSV}")

# --- Write one TEI file per article -----------------------------------------
# The div order matters: div[1] = head, div[2] = subhead, div[3] = content.
# Block 6 relies on it to work out where an annotated passage sits.

def safe_text(value):
    return "" if pd.isna(value) else str(value).strip()


output_dir = Path(TEI_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
NSMAP = {None: "http://www.tei-c.org/ns/1.0"}

for _, row in sample.iterrows():
    article_id = safe_text(row["content_id"])
    if not article_id:
        continue

    tei = etree.Element("TEI", nsmap=NSMAP)

    header = etree.SubElement(tei, "teiHeader")
    file_desc = etree.SubElement(header, "fileDesc")
    title_stmt = etree.SubElement(file_desc, "titleStmt")
    etree.SubElement(title_stmt, "title").text = safe_text(row.get("head"))

    body = etree.SubElement(etree.SubElement(tei, "text"), "body")
    etree.SubElement(body, "div", type="head").text = safe_text(row.get("head"))
    etree.SubElement(body, "div", type="subhead").text = safe_text(row.get("subhead"))
    etree.SubElement(body, "div", type="content").text = safe_text(row.get("content"))

    etree.ElementTree(tei).write(
        output_dir / f"{article_id}.xml",
        encoding="utf-8", xml_declaration=True, pretty_print=True,
    )

print(f"  TEI files written -> {TEI_DIR}/")
print()
print("Next steps:")
print("  1. Import the TEI files into your annotation tool and annotate them.")
print(f"  2. Export the result as CSV into '{ANNOTATION_DIR}/'.")
print(f"     Name it so it matches the pattern: {ANNOTATION_PATTERNS['impacts'].format(lang=LANGUAGE)}")
print(f"     For example: {LANGUAGE}_annotated_impacts_round3.csv")
print("  3. Re-run Block 4 and everything after it.")

---
# Appendix B — Agreement between two annotators

**Not part of a normal run.** Use this when two people have annotated the same articles
and you want to report how consistently they applied the guidelines.

Cohen's kappa measures agreement corrected for what you would expect by chance. As a rough
guide: above 0.8 is very good, 0.6–0.8 is solid, below 0.4 means the guidelines need work.

Two versions are computed:

- **Strict** — the two annotators must have highlighted exactly the same character range.
- **Lenient** — highlights within 5 characters of each other count as the same passage.
  This separates genuine disagreement about the *label* from trivial disagreement about
  where a sentence starts.

In [3]:
# ============================================================================
# APPENDIX B — INTER-ANNOTATOR AGREEMENT
# ============================================================================

from sklearn.metrics import cohen_kappa_score

# --- Point these at the two annotators' export files ------------------------
ANNOTATOR_A_FILE = os.path.join(ANNOTATION_DIR, f"{LANGUAGE}_annotator_A.csv")
ANNOTATOR_B_FILE = os.path.join(ANNOTATION_DIR, f"{LANGUAGE}_annotator_B.csv")

EXCLUDE_LABEL = "general_statement"   # label to leave out of the comparison
TOLERANCE = 5                          # characters, for the lenient version

for path in (ANNOTATOR_A_FILE, ANNOTATOR_B_FILE):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"'{path}' not found. Edit ANNOTATOR_A_FILE and ANNOTATOR_B_FILE above to "
            "point at the two files you want to compare."
        )

A = pd.read_csv(ANNOTATOR_A_FILE).replace("", pd.NA)
B = pd.read_csv(ANNOTATOR_B_FILE).replace("", pd.NA)

# Compare category annotations only, not place annotations.
def category_spans(frame):
    if "TYPE" in frame.columns:
        return frame[frame["TYPE"].isna() & frame["TAGS"].notna()].copy()
    return frame[frame["TAGS"].notna()].copy()


cat_A, cat_B = category_spans(A), category_spans(B)

# ---------------------------------------------------------------------------
# Strict: the character ranges must match exactly
# ---------------------------------------------------------------------------
merged = pd.merge(cat_A, cat_B, on=["FILE", "ANCHOR"], how="outer", suffixes=("_A", "_B"))
merged["TAGS_A"] = merged["TAGS_A"].fillna("NO_TAG")
merged["TAGS_B"] = merged["TAGS_B"].fillna("NO_TAG")

if EXCLUDE_LABEL:
    merged = merged[(merged["TAGS_A"] != EXCLUDE_LABEL) & (merged["TAGS_B"] != EXCLUDE_LABEL)]

kappa_strict = cohen_kappa_score(merged["TAGS_A"], merged["TAGS_B"])

print("=" * 70)
print("INTER-ANNOTATOR AGREEMENT")
print("=" * 70)
print(f"  passages compared:  {len(merged):,}")
print(f"  Cohen's kappa (exact span match):   {kappa_strict:.3f}")

# ---------------------------------------------------------------------------
# Lenient: allow the spans to differ by a few characters
# ---------------------------------------------------------------------------
def extract_range(anchor):
    if pd.isna(anchor):
        return None, None
    numbers = re.findall(r"::(\d+)", str(anchor))
    if len(numbers) != 2:
        return None, None
    return int(numbers[0]), int(numbers[1])


for frame in (cat_A, cat_B):
    frame[["START", "END"]] = frame["ANCHOR"].apply(lambda x: pd.Series(extract_range(x)))

pairs = []
for file_id in pd.concat([cat_A["FILE"], cat_B["FILE"]]).unique():
    a_rows = cat_A[cat_A["FILE"] == file_id]
    b_rows = cat_B[cat_B["FILE"] == file_id]
    used_b = set()

    for _, a in a_rows.iterrows():
        candidates = b_rows[
            ((b_rows["START"] - a["START"]).abs() <= TOLERANCE) |
            ((b_rows["END"] - a["END"]).abs() <= TOLERANCE)
        ]
        candidates = candidates[~candidates.index.isin(used_b)]

        if not candidates.empty:
            candidates = candidates.assign(distance=(
                (candidates["START"] - a["START"]).abs() +
                (candidates["END"] - a["END"]).abs()
            ))
            b = candidates.sort_values("distance").iloc[0]
            used_b.add(b.name)
            pairs.append({"TAGS_A": a["TAGS"], "TAGS_B": b["TAGS"]})
        else:
            pairs.append({"TAGS_A": a["TAGS"], "TAGS_B": "NO_TAG"})

    for _, b in b_rows.loc[~b_rows.index.isin(used_b)].iterrows():
        pairs.append({"TAGS_A": "NO_TAG", "TAGS_B": b["TAGS"]})

lenient = pd.DataFrame(pairs)
if EXCLUDE_LABEL:
    lenient = lenient[(lenient["TAGS_A"] != EXCLUDE_LABEL) & (lenient["TAGS_B"] != EXCLUDE_LABEL)]

kappa_lenient = cohen_kappa_score(lenient["TAGS_A"], lenient["TAGS_B"])
print(f"  Cohen's kappa (within {TOLERANCE} characters): {kappa_lenient:.3f}")
print("=" * 70)
print("\n  Guide: >0.80 very good, 0.60-0.80 solid, 0.40-0.60 moderate, <0.40 poor")

NameError: name 'os' is not defined